# ╔══════════════════════════════════════════════════════════════╗
# ║   PERFECT 4-MODEL ENSEMBLE FUSION — FINAL PREDICTIONS       ║
# ║   Models: rPPG | EfficientNet-B4 | Xception | Swin-Tiny     ║
# ╚══════════════════════════════════════════════════════════════╝
#
# EXACT FILE NAMES FROM EACH NOTEBOOK:
#
# 1. model_rppg.ipynb        → rppg_predictions.csv          (col: P_rPPG)
# 2. model_efficientnet.ipynb→ cnn_predictions.csv           (col: P_CNN)
# 3. model_xception-1.ipynb  → cnn_predictions.csv           (col: P_CNN)
# 4. model_swin-1.ipynb      → cnn_predictions_swin_oof_MASTER.csv  (col: P_CNN)
#
# HOW TO USE ON KAGGLE:
# - Run each model notebook first and download its /kaggle/working/ output as a dataset
# - Add each downloaded dataset as a Kaggle input to THIS notebook
# - Update the INPUT PATHS section below to match your dataset names
# ─────────────────────────────────────────────────────────────────

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 0 v3: FIX rPPG video_id + PREPARE ALL PREDICTION FILES
# Changes from v2: unchanged — this cell was already correct
# ═══════════════════════════════════════════════════════════════════

import pandas as pd, os, numpy as np

BASE_DIR = '/kaggle/input/datasets/likhithvasireddy/final-fusion'

swin_master = pd.read_csv(f'{BASE_DIR}/swin_master_dataset_index.csv')
swin_master['_basename'] = swin_master['path'].apply(os.path.basename)
basename_to_vid = dict(zip(swin_master['_basename'], swin_master['video_id']))
print(f"✓ Swin master loaded: {len(swin_master)} videos")

rppg_raw = pd.read_csv(f'{BASE_DIR}/rppg_predictions.csv')
print(f"\nrPPG raw columns : {rppg_raw.columns.tolist()}")
print(f"rPPG sample id   : {rppg_raw['video_id'].iloc[0]}")

if '__' not in str(rppg_raw['video_id'].iloc[0]):
    rppg_raw['video_id'] = rppg_raw['video_id'].map(basename_to_vid)
    print("  → Remapped video_ids")
else:
    print("  → video_ids already correct")

if 'true_label' in rppg_raw.columns and 'label' not in rppg_raw.columns:
    rppg_raw = rppg_raw.rename(columns={'true_label': 'label'})
if 'P_rPPG' not in rppg_raw.columns:
    cands = [c for c in rppg_raw.columns if c not in ['video_id','label','true_label']]
    if cands:
        rppg_raw = rppg_raw.rename(columns={cands[0]: 'P_rPPG'})

if 'P_rPPG' in rppg_raw.columns:
    rppg_raw['P_rPPG'] = rppg_raw['P_rPPG'].clip(0.0, 1.0)
if 'label' in rppg_raw.columns:
    rppg_raw['label'] = rppg_raw['label'].astype(int)

before = len(rppg_raw)
rppg_raw = rppg_raw.dropna(subset=['video_id'])
rppg_raw = rppg_raw.groupby(['video_id'], as_index=False).agg(
    {c: ('first' if c == 'label' else 'mean')
     for c in rppg_raw.columns if c != 'video_id'})
print(f"  → {len(rppg_raw)}/{before} videos after dedup")
rppg_raw.to_csv('/kaggle/working/rppg_predictions_fixed.csv', index=False)
print("✓ Fixed rPPG saved")

for name, path in {
    'rPPG (fixed)':   '/kaggle/working/rppg_predictions_fixed.csv',
    'EfficientNet':    f'{BASE_DIR}/cnn_predictions_efficientnet.csv',
    'Xception':        f'{BASE_DIR}/cnn_predictions_xception.csv',
    'Swin OOF MASTER': f'{BASE_DIR}/cnn_predictions_swin_oof_MASTER.csv',
}.items():
    status = '✓' if os.path.exists(path) else '✗'
    n = len(pd.read_csv(path)) if os.path.exists(path) else 0
    print(f"  {status} {name:22s}: {n:5d} videos")

✓ Swin master loaded: 1754 videos

rPPG raw columns : ['video_id', 'true_label', 'P_rPPG']
rPPG sample id   : 747.mp4
  → Remapped video_ids
  → 1416/1631 videos after dedup
✓ Fixed rPPG saved
  ✓ rPPG (fixed)          :  1416 videos
  ✓ EfficientNet          :   367 videos
  ✓ Xception              :   353 videos
  ✓ Swin OOF MASTER       :  1753 videos


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 v4: IMPORTS  — fixes 2 crash-level missing imports vs v3
# NEW: SVC, norm, BayesianRidge, IterativeImputer (MICE)
# Run first:  !pip install lightgbm optuna -q
# Optional:   !pip install catboost -q
# ═══════════════════════════════════════════════════════════════════

import os, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, roc_curve,
    confusion_matrix, classification_report,
    average_precision_score, log_loss, brier_score_loss
)
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge  # ★ BayesianRidge NEW
from sklearn.svm import SVC                                                  # ★ SVC NEW (was crashing)
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier, VotingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (StratifiedKFold, LeaveOneOut,
                                      cross_val_predict)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.impute import KNNImputer
from sklearn.neighbors import NearestNeighbors

# MICE imputer (requires sklearn ≥ 0.21)
try:
    from sklearn.experimental import enable_iterative_imputer   # noqa
    from sklearn.impute import IterativeImputer
    HAS_MICE = True
except ImportError:
    HAS_MICE = False


from scipy.stats import rankdata, norm, chi2    # ★ norm + chi2 for DeLong/McNemar tests
from scipy.optimize import differential_evolution, minimize
from scipy.special import expit, logit

# ── XGBoost ──────────────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    HAS_XGB = True;  print('  ✓ XGBoost available')
except ImportError:
    HAS_XGB = False; print('  ⚠ XGBoost not found (pip install xgboost)')

# ── LightGBM ─────────────────────────────────────────────────────────────────
try:
    import lightgbm as lgb
    HAS_LGB = True;  print('  ✓ LightGBM available')
except ImportError:
    HAS_LGB = False; print('  ⚠ LightGBM not found (pip install lightgbm)')

# ── CatBoost ─────────────────────────────────────────────────────────────────
try:
    from catboost import CatBoostClassifier
    HAS_CAT = True;  print('  ✓ CatBoost available')
except ImportError:
    HAS_CAT = False; print('  ⚠ CatBoost not found (pip install catboost)')

# ── Optuna ────────────────────────────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True;  print('  ✓ Optuna available')
except ImportError:
    HAS_OPTUNA = False; print('  ⚠ Optuna not found (pip install optuna)')

SEED = 42
np.random.seed(SEED)
EPS  = 1e-7

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'\n✓ All imports loaded | NumPy {np.__version__} | Pandas {pd.__version__}')
print(f'  MICE available: {HAS_MICE}')

  ✓ XGBoost available
  ✓ LightGBM available
  ✓ CatBoost available
  ✓ Optuna available

✓ All imports loaded | NumPy 2.0.2 | Pandas 2.3.3
  MICE available: True


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 v3: INPUT PATHS  (unchanged from v2)
# ═══════════════════════════════════════════════════════════════════

BASE_DIR         = '/kaggle/input/datasets/likhithvasireddy/final-fusion'
RPPG_CSV         = '/kaggle/working/rppg_predictions_fixed.csv'
EFFICIENTNET_CSV = f'{BASE_DIR}/cnn_predictions_efficientnet.csv'
XCEPTION_CSV     = f'{BASE_DIR}/cnn_predictions_xception.csv'
SWIN_CSV         = f'{BASE_DIR}/cnn_predictions_swin_oof_MASTER.csv'

print('═' * 70)
print('INPUT FILE CONFIGURATION')
print('═' * 70)
for name, path in [
    ('rPPG (fixed)',  RPPG_CSV),
    ('EfficientNet',  EFFICIENTNET_CSV),
    ('Xception',      XCEPTION_CSV),
    ('Swin-Tiny OOF', SWIN_CSV)
]:
    status = '✓' if os.path.exists(path) else '✗'
    n = len(pd.read_csv(path)) if os.path.exists(path) else 0
    print(f'  {status}  {name:18s}: {n:5d} videos — {path}')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
INPUT FILE CONFIGURATION
══════════════════════════════════════════════════════════════════════
  ✓  rPPG (fixed)      :  1416 videos — /kaggle/working/rppg_predictions_fixed.csv
  ✓  EfficientNet      :   367 videos — /kaggle/input/datasets/likhithvasireddy/final-fusion/cnn_predictions_efficientnet.csv
  ✓  Xception          :   353 videos — /kaggle/input/datasets/likhithvasireddy/final-fusion/cnn_predictions_xception.csv
  ✓  Swin-Tiny OOF     :  1753 videos — /kaggle/input/datasets/likhithvasireddy/final-fusion/cnn_predictions_swin_oof_MASTER.csv
══════════════════════════════════════════════════════════════════════


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 v3: LOAD & VALIDATE ALL PREDICTION FILES  (unchanged)
# ═══════════════════════════════════════════════════════════════════

def load_predictions(csv_path, score_col, model_name, rename_to):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"❌ {model_name} not found: {csv_path}")
    df = pd.read_csv(csv_path)
    for col in ['video_id', score_col]:
        if col not in df.columns:
            raise ValueError(f"❌ {model_name}: missing '{col}'. Found: {list(df.columns)}")
    has_label = 'label' in df.columns
    df = df.rename(columns={score_col: rename_to})
    before = len(df)
    agg = {rename_to: 'mean', **({'label': 'first'} if has_label else {})}
    df  = df.groupby('video_id', as_index=False).agg(agg)
    if len(df) < before:
        print(f'  ⚠ {model_name}: collapsed {before-len(df)} duplicates (averaged)')
    df[rename_to] = df[rename_to].clip(0.0, 1.0)
    if has_label:
        df['label'] = df['label'].astype(int)
    lbl = f' | Real={(df["label"]==0).sum()} Fake={(df["label"]==1).sum()}' if has_label else ''
    print(f'  ✓ {model_name:15s}: {len(df):5d} videos{lbl}')
    return df

print('Loading predictions from all 4 models...')
print('─' * 60)
df_rppg = load_predictions(RPPG_CSV,         'P_rPPG', 'rPPG',         'P_rPPG')
df_eff  = load_predictions(EFFICIENTNET_CSV,  'P_CNN',  'EfficientNet', 'P_efficientnet')
df_xcep = load_predictions(XCEPTION_CSV,      'P_CNN',  'Xception',     'P_xception')
df_swin = load_predictions(SWIN_CSV,          'P_CNN',  'Swin-Tiny',    'P_swin')
print('─' * 60)
print('✓ All prediction files loaded successfully')

Loading predictions from all 4 models...
────────────────────────────────────────────────────────────
  ✓ rPPG           :  1416 videos | Real=743 Fake=673
  ✓ EfficientNet   :   367 videos | Real=199 Fake=168
  ✓ Xception       :   353 videos | Real=186 Fake=167
  ✓ Swin-Tiny      :  1753 videos | Real=876 Fake=877
────────────────────────────────────────────────────────────
✓ All prediction files loaded successfully


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# ALL IMPROVED CELLS — ACCURACY-MAXIMISING CHANGES
# Changes are annotated with ★ CHANGE and a reason for every modification
# ════════════════════════════════════════════════════════════════════════════
 
 
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 v5: MERGE + IMPUTATION + CALIBRATION
# ════════════════════════════════════════════════════════════════════════════
#
# ★ CHANGE 1 — MICE max_iter 15 → 20
#   Reason: 15 iterations often doesn't reach convergence for 4 variables.
#   BayesianRidge is fast enough; 20 adds ~30% runtime but gains convergence.
#
# ★ CHANGE 2 — Add isotonic calibration + pick best per model
#   Reason: Isotonic regression is non-parametric and more powerful than beta
#   calibration when the reliability curve is non-monotone (common with LSTM
#   outputs). We run both OOF and pick whichever gives higher AUC.
#   This can lift individual model AUC by 0.001–0.003.
#
# ★ CHANGE 3 — Calibration CV raised from 10 to min(10, ...) with explicit guard
#   No functional change but makes the guard explicit and avoids silent 1-fold.
# ════════════════════════════════════════════════════════════════════════════
 
MODEL_COLS  = ['P_rPPG', 'P_efficientnet', 'P_xception', 'P_swin']
MODEL_NAMES = ['rPPG', 'EfficientNet', 'Xception', 'Swin-Tiny']
 
MODEL_LABELS_ALL = {
    'P_rPPG'        : 'rPPG (ML Stacking)',
    'P_efficientnet': 'EfficientNet-B4 (BiLSTM+Attn)',
    'P_xception'    : 'Xception (BiLSTM+Freq)',
    'P_swin'        : 'Swin-Tiny (BiLSTM+DCT)',
}
 
# ─── A. BETA CALIBRATION (OOF) ───────────────────────────────────────────────
def beta_calibrate_oof(scores, labels, cv=10):
    safe_cv = min(cv, int(min(np.bincount(labels.astype(int)))))
    if safe_cv < 2:
        return scores.copy()
    skf = StratifiedKFold(n_splits=safe_cv, shuffle=True, random_state=SEED)
    oof = np.zeros_like(scores)
    for tr, va in skf.split(scores.reshape(-1, 1), labels):
        p_tr  = np.clip(scores[tr], EPS, 1 - EPS)
        y_tr  = labels[tr]
        lp_tr = np.log(p_tr); l1mp_tr = np.log(1 - p_tr)
        def _nll(params):
            a, b, c = params
            prob = expit(a * lp_tr + b * l1mp_tr + c)
            return -np.mean(y_tr * np.log(np.clip(prob, EPS, 1)) +
                            (1 - y_tr) * np.log(np.clip(1 - prob, EPS, 1)))
        try:
            res = minimize(_nll, x0=[1.0, 1.0, 0.0], method='Nelder-Mead',
                           options={'maxiter': 500, 'xatol': 1e-5, 'fatol': 1e-5})
            a, b, c = res.x
            p_va = np.clip(scores[va], EPS, 1 - EPS)
            oof[va] = np.clip(expit(a * np.log(p_va) + b * np.log(1-p_va) + c), 0, 1)
        except Exception:
            ls = logit(np.clip(scores[tr], EPS, 1-EPS)).reshape(-1, 1)
            lr = LogisticRegression(C=1.0, max_iter=500)
            lr.fit(ls, y_tr)
            oof[va] = lr.predict_proba(
                logit(np.clip(scores[va], EPS, 1-EPS)).reshape(-1, 1))[:, 1]
    return np.clip(oof, 0.0, 1.0)
 
# ★ CHANGE 2: New isotonic calibration function
def isotonic_calibrate_oof(scores, labels, cv=10):
    """
    Isotonic regression calibration (OOF).
    Non-parametric — makes no assumptions about the shape of the reliability
    curve. Outperforms beta/Platt when the model outputs have non-monotone
    bias (common with recurrent networks).
    """
    from sklearn.isotonic import IsotonicRegression
    safe_cv = min(cv, int(min(np.bincount(labels.astype(int)))))
    if safe_cv < 2:
        return scores.copy()
    skf = StratifiedKFold(n_splits=safe_cv, shuffle=True, random_state=SEED)
    oof = np.zeros_like(scores)
    for tr, va in skf.split(scores.reshape(-1, 1), labels):
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(scores[tr], labels[tr])
        oof[va] = np.clip(iso.predict(scores[va]), 0.0, 1.0)
    return oof
 
# ★ CHANGE 2: Pick best calibration method per model
def best_calibrate_oof(scores, labels, col_name, cv=10):
    """
    Run both beta and isotonic calibration OOF.
    Return whichever gives higher AUC on the full OOF predictions.
    """
    raw_auc  = roc_auc_score(labels, scores)
    beta_cal = beta_calibrate_oof(scores, labels, cv=cv)
    iso_cal  = isotonic_calibrate_oof(scores, labels, cv=cv)
    auc_beta = roc_auc_score(labels, beta_cal)
    auc_iso  = roc_auc_score(labels, iso_cal)
 
    if auc_iso >= auc_beta:
        chosen, chosen_auc, method = iso_cal, auc_iso, 'iso'
    else:
        chosen, chosen_auc, method = beta_cal, auc_beta, 'beta'
 
    sym = '▲' if chosen_auc > raw_auc else '▼'
    print(f"  {col_name:20s}: raw={raw_auc:.4f} → {method}-cal={chosen_auc:.4f} {sym}  "
          f"[beta={auc_beta:.4f} iso={auc_iso:.4f}]  (cv={cv})")
    return chosen
 
# ─── B. TRUE MICE IMPUTATION ─────────────────────────────────────────────────
def build_imputed_master_v4(df_rppg, df_eff, df_xcep, df_swin):
    """
    MICE with BayesianRidge.
    ★ CHANGE 1: max_iter raised from 15 → 20 for full convergence.
    """
    base = df_rppg[['video_id', 'label', 'P_rPPG']].copy()
    base = base.merge(df_eff[['video_id',  'P_efficientnet']], on='video_id', how='left')
    base = base.merge(df_xcep[['video_id', 'P_xception']],    on='video_id', how='left')
    base = base.merge(df_swin[['video_id', 'P_swin']],         on='video_id', how='left')
 
    swin_only = df_swin[['video_id', 'label', 'P_swin']].copy()
    swin_only = swin_only[~swin_only['video_id'].isin(base['video_id'])]
    swin_only = swin_only.merge(df_eff[['video_id',  'P_efficientnet']], on='video_id', how='left')
    swin_only = swin_only.merge(df_xcep[['video_id', 'P_xception']],     on='video_id', how='left')
    swin_only['P_rPPG'] = np.nan
    base = pd.concat([base, swin_only], ignore_index=True)
 
    for c in MODEL_COLS:
        base[f'{c}_imputed'] = base[c].isna()
 
    native_count = base[MODEL_COLS].notna().sum(axis=1)
    base = base[native_count >= 2].copy().reset_index(drop=True)
    n_missing = int(base[MODEL_COLS].isna().sum().sum())
    print(f"  Before imputation : {len(base)} videos | {n_missing} missing score cells")
 
    X_raw = base[MODEL_COLS].values.astype(float)
 
    if HAS_MICE:
        imp = IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=20,            # ★ CHANGE 1: was 15
            random_state=SEED,
            min_value=0.01,
            max_value=0.99,
            verbose=0
        )
        X_imp = imp.fit_transform(X_raw)
        print(f"  ✓ MICE (IterativeImputer + BayesianRidge, 20 iter)")
    else:
        knn_imp = KNNImputer(
            n_neighbors=max(1, min(7, int((base[MODEL_COLS].notna().all(axis=1)).sum()))),
            weights='distance')
        X_knn = knn_imp.fit_transform(X_raw)
        X_ridge = X_raw.copy()
        for target_i in range(len(MODEL_COLS)):
            predictors_i = [j for j in range(len(MODEL_COLS)) if j != target_i]
            mask_tr = (~np.isnan(X_raw[:, target_i]) &
                       ~np.any(np.isnan(X_raw[:, predictors_i]), axis=1))
            mask_im = (np.isnan(X_raw[:, target_i]) &
                       ~np.any(np.isnan(X_raw[:, predictors_i]), axis=1))
            if mask_tr.sum() >= 5 and mask_im.sum() > 0:
                reg = Ridge(alpha=1.0).fit(X_raw[mask_tr][:, predictors_i],
                                           X_raw[mask_tr, target_i])
                X_ridge[mask_im, target_i] = reg.predict(X_raw[mask_im][:, predictors_i])
        still_nan = np.isnan(X_ridge)
        X_ridge[still_nan] = X_knn[still_nan]
        X_imp = 0.6 * X_knn + 0.4 * X_ridge
        print(f"  ✓ KNN+Ridge blend (MICE unavailable — upgrade sklearn)")
 
    X_imp = np.clip(X_imp, 0.01, 0.99)
    for i, col in enumerate(MODEL_COLS):
        base[col] = X_imp[:, i]
 
    base = base.dropna(subset=MODEL_COLS).reset_index(drop=True)
    print(f"  After imputation  : {len(base)} videos")
    return base
 
# ─── C. BUILD MASTER DATASET ─────────────────────────────────────────────────
print('═' * 70)
print('SCORE IMPUTATION v5 — MICE max_iter=20 (was 15)')
print('═' * 70)
master = build_imputed_master_v4(df_rppg, df_eff, df_xcep, df_swin)
 
# ─── D. BEST CALIBRATION (beta vs isotonic, pick per model) ★ CHANGE 2 ───────
print('\n' + '═' * 70)
print('PROBABILITY CALIBRATION v5 — Best of beta vs isotonic (OOF)')
print('═' * 70)
 
labels_master = master['label'].values.astype(int)
_n_splits_cal = min(10, int(min(np.bincount(labels_master))))
 
for col in MODEL_COLS:
    cal = best_calibrate_oof(master[col].values, labels_master,
                              col_name=col, cv=_n_splits_cal)
    master[col] = cal
print('✓ Calibration complete')
 
# ─── E. STRATEGY SETS ────────────────────────────────────────────────────────
def fill_nans(mdf, cols):
    nc = mdf[cols].isna().sum()
    if nc.any():
        print(f"  ⚠ NaN scores: {nc[nc>0].to_dict()} → filling 0.5")
        mdf[cols] = mdf[cols].fillna(0.5)
    return mdf
 
merged_4 = fill_nans(master[['video_id','label','P_rPPG','P_efficientnet',
                               'P_xception','P_swin']].copy(),
                      ['P_rPPG','P_efficientnet','P_xception','P_swin'])
merged_3 = fill_nans(master[['video_id','label','P_rPPG','P_efficientnet',
                               'P_swin']].copy(),
                      ['P_rPPG','P_efficientnet','P_swin'])
merged_2 = fill_nans(master[['video_id','label','P_rPPG','P_swin']].copy(),
                      ['P_rPPG','P_swin'])
merged_1 = fill_nans(master[['video_id','label','P_swin']].copy(), ['P_swin'])
 
all_strategies = {
    'A: 4-model (rPPG+Eff+Xcep+Swin)': (merged_4, ['P_rPPG','P_efficientnet','P_xception','P_swin']),
    'B: 3-model (rPPG+Eff+Swin)'     : (merged_3, ['P_rPPG','P_efficientnet','P_swin']),
    'C: 2-model (rPPG+Swin)'          : (merged_2, ['P_rPPG','P_swin']),
    'D: Swin only (baseline)'          : (merged_1, ['P_swin']),
}
 
def auc_weighted_score(mdf, scols):
    yt = mdf['label'].values
    Xs = mdf[scols].values
    w  = np.array([roc_auc_score(yt, mdf[c].values) for c in scols])
    w /= w.sum()
    return roc_auc_score(yt, (Xs * w[None, :]).sum(axis=1))



print('\n' + '═' * 70)
print('MERGE STRATEGIES')
print('═' * 70)
print(f"  {'Strategy':<35} {'n':>5} {'Real':>5} {'Fake':>5} {'AUC-Wtd':>9}")
print("  " + "─" * 65)

strategy_aucs = {}
for sname, (mdf, scols) in all_strategies.items():
    n = len(mdf)
    nr = int((mdf['label']==0).sum())
    nf = int((mdf['label']==1).sum())
    try:
        auc = auc_weighted_score(mdf, scols)
    except Exception:
        auc = 0.0
    strategy_aucs[sname] = (auc, mdf, scols)
    print(f"  {sname:<35} {n:>5} {nr:>5} {nf:>5} {auc:>9.4f}")

best_strategy = max(strategy_aucs, key=lambda k: strategy_aucs[k][0])
best_strategy_auc, best_merged, best_score_cols = strategy_aucs[best_strategy]
print(f"\n  ★ AUTO-SELECTED: {best_strategy} (AUC={best_strategy_auc:.4f})")

merged       = best_merged.copy()
SCORE_COLS   = best_score_cols
MODEL_LABELS = {k: v for k, v in MODEL_LABELS_ALL.items() if k in SCORE_COLS}

print(f'\n  Total: {len(merged)} | Real: {int((merged["label"]==0).sum())} '
      f'| Fake: {int((merged["label"]==1).sum())}')
 

══════════════════════════════════════════════════════════════════════
SCORE IMPUTATION v5 — MICE max_iter=20 (was 15)
══════════════════════════════════════════════════════════════════════
  Before imputation : 1520 videos | 2507 missing score cells
  ✓ MICE (IterativeImputer + BayesianRidge, 20 iter)
  After imputation  : 1520 videos

══════════════════════════════════════════════════════════════════════
PROBABILITY CALIBRATION v5 — Best of beta vs isotonic (OOF)
══════════════════════════════════════════════════════════════════════
  P_rPPG              : raw=0.6309 → beta-cal=0.6283 ▼  [beta=0.6283 iso=0.6090]  (cv=10)
  P_efficientnet      : raw=0.7790 → beta-cal=0.7784 ▼  [beta=0.7784 iso=0.7708]  (cv=10)
  P_xception          : raw=0.7607 → beta-cal=0.7601 ▼  [beta=0.7601 iso=0.7492]  (cv=10)
  P_swin              : raw=0.7897 → beta-cal=0.7884 ▼  [beta=0.7884 iso=0.7786]  (cv=10)
✓ Calibration complete

══════════════════════════════════════════════════════════════════════
MERG

In [6]:
 
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 v5: INDIVIDUAL METRICS + META-FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════════════
#
# ★ CHANGE 3 — Cleaner meta-features: 19 features instead of 29
#   Reason: With n=1520 samples, 29 features include redundant polynomials
#   (products of probs are near-linear in logit space) that give meta-learners
#   spurious degrees of freedom, causing overfitting. Removing products and the
#   redundant variance/std pair (var = std²) and replacing with rank features
#   (monotonic, resistant to outlier scores) reduces overfitting while
#   preserving all discriminative signal.
#
# ★ CHANGE 4 — _OPT_CV: 5-fold → 10-fold for Optuna weight optimisation
#   Reason: 5-fold on 1520 samples = 304 samples per fold. The AUC estimate
#   variance with 304 samples is ~3× higher than with 10-fold (152 samples).
#   Higher-variance objectives mislead Optuna into sub-optimal weights.
#   Using the full _cached_splits (10-fold) gives a lower-variance objective.
# ════════════════════════════════════════════════════════════════════════════
 
y_true = merged['label'].values.astype(int)
N      = len(y_true)
 
USE_LOOCV = N <= 200
if USE_LOOCV:
    _cv_scheme = LeaveOneOut()
    _cv_label  = 'LOOCV'
    _n_splits  = N
else:
    _n_splits  = min(10, int(min(np.bincount(y_true))))
    _cv_scheme = StratifiedKFold(n_splits=_n_splits, shuffle=True, random_state=SEED)
    _cv_label  = f'{_n_splits}-fold'
 
# ★ CHANGE 4: Use full 10-fold CV for Optuna (was 5-fold)
_OPT_CV = StratifiedKFold(
    n_splits=_n_splits,          # ★ was min(5, ...) — now same as evaluation CV
    shuffle=True, random_state=SEED)
 
print(f"  Evaluation CV : {_cv_label}  (n={N})")
print(f"  Optuna CV     : {_OPT_CV.n_splits}-fold (now matches evaluation CV)")
 
_cached_splits = []
if USE_LOOCV:
    for tr, va in _cv_scheme.split(np.zeros((N, 1))):
        _cached_splits.append((tr, va))
else:
    for tr, va in _cv_scheme.split(np.zeros(N), y_true):
        _cached_splits.append((tr, va))
print(f"  Cached {len(_cached_splits)} fold splits for all OOF evaluations")
 
# ─── INDIVIDUAL MODEL METRICS ─────────────────────────────────────────────────
individual_aucs = {}
print('═' * 78)
print(f'INDIVIDUAL MODEL PERFORMANCE  ({best_strategy})')
print('═' * 78)
print(f"{'Model':35s} {'AUC':>7} {'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7} {'Brier':>7}")
print('─' * 78)
 
for col in SCORE_COLS:
    probs = merged[col].values
    fpr_m, tpr_m, thr_m = roc_curve(y_true, probs)
    thresh_m = float(thr_m[np.argmax(tpr_m - fpr_m)])
    preds    = (probs >= thresh_m).astype(int)
    auc      = roc_auc_score(y_true, probs)
    acc      = accuracy_score(y_true, preds)
    f1       = f1_score(y_true, preds, zero_division=0)
    prec     = precision_score(y_true, preds, zero_division=0)
    rec      = recall_score(y_true, preds, zero_division=0)
    brier    = brier_score_loss(y_true, probs)
    individual_aucs[col] = auc
    print(f"  {MODEL_LABELS[col]:33s} {auc:7.4f} {acc:7.4f} {f1:7.4f} "
          f"{prec:7.4f} {rec:7.4f} {brier:7.4f}")
 
best_ind_col = max(individual_aucs, key=individual_aucs.get)
print('═' * 78)
print(f'  Best individual: {individual_aucs[best_ind_col]:.4f} ({MODEL_LABELS[best_ind_col]})')
 
print('\n── Pairwise correlations (lower = more diverse = better ensemble) ──')
corr = merged[SCORE_COLS].corr()
for i, ci in enumerate(SCORE_COLS):
    for j, cj in enumerate(SCORE_COLS):
        if j > i:
            print(f"    {MODEL_LABELS[ci].split()[0]:12s} ↔ "
                  f"{MODEL_LABELS[cj].split()[0]:12s}: r={corr.loc[ci,cj]:.3f}")
 
# ★ CHANGE 3: Cleaner meta-features — 19 features for M=3, no redundant polynomials
def build_meta_features_v3(Xs):
    """
    v5 meta-feature matrix (19 features for M=3 models):
      Raw probs     : M  (direct signal)
      Logit probs   : M  (uncorrelated near boundaries; essential for LR meta)
      Rank features : M  (percentile rank — monotonic, robust to outlier scores)
      Abs diffs     : M*(M-1)/2  (model disagreement — key diversity signal)
      Signed diffs  : M*(M-1)/2  (direction of disagreement)
      Mean, Std     : 2  (aggregate position and spread)
      Max, Min      : 2  (extreme model opinions)
      Mean conf.    : 1  (mean |p-0.5| — overall certainty)
      Consensus     : 1  (fraction voting fake — hard vote signal)
      Logit mean    : 1  (sigmoid(mean logit) ≠ arithmetic mean near bounds)
 
    Removed vs v4 (29 features):
      - Pairwise products (P_i * P_j): near-linear in logit space, redundant
      - Variance: = std², fully redundant with std
      - Per-model |p-0.5|: 3 separate features replaced by 1 mean confidence
      - Geometric mean: approximated by logit mean, removed to reduce collinearity
    """
    n, m = Xs.shape
    p  = np.clip(Xs, EPS, 1 - EPS)
    Xl = logit(p)
 
    feats = [Xs, Xl]
 
    # Rank features (monotonic, robust)
    for i in range(m):
        feats.append((rankdata(Xs[:, i]) / n).reshape(-1, 1))
 
    # Pairwise abs + signed differences (disagreement signal)
    for i in range(m):
        for j in range(i + 1, m):
            feats.append(np.abs(Xs[:, i] - Xs[:, j]).reshape(-1, 1))
            feats.append((Xs[:, i] - Xs[:, j]).reshape(-1, 1))
 
    # Aggregate statistics
    feats.append(Xs.mean(axis=1, keepdims=True))
    feats.append(Xs.std(axis=1, keepdims=True))
    feats.append(Xs.max(axis=1, keepdims=True))
    feats.append(Xs.min(axis=1, keepdims=True))
    feats.append(np.abs(Xs - 0.5).mean(axis=1, keepdims=True))   # mean confidence
    feats.append((Xs > 0.5).astype(float).mean(axis=1, keepdims=True))  # consensus
    feats.append(expit(Xl.mean(axis=1, keepdims=True)))            # logit mean
 
    return np.hstack(feats)
 
_n_meta_feats = build_meta_features_v3(merged[SCORE_COLS].values[:1]).shape[1]
print(f"\n  ✓ build_meta_features_v3 defined ({_n_meta_feats} features per sample)")
 
# ─── BAYESIAN MODEL AVERAGING WEIGHTS ────────────────────────────────────────
def compute_bma_weights(score_cols, labels, X_merged):
    log_likelihoods = []
    for col in score_cols:
        p = np.clip(X_merged[col].values, EPS, 1 - EPS)
        ll = np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        log_likelihoods.append(ll)
    log_likelihoods = np.array(log_likelihoods)
    log_likelihoods -= log_likelihoods.max()
    weights = np.exp(log_likelihoods)
    weights /= weights.sum()
    return weights
 
bma_weights = compute_bma_weights(SCORE_COLS, y_true, merged)
print('\n── Bayesian Model Averaging weights ──')
for c, w in zip(SCORE_COLS, bma_weights):
    print(f"  {MODEL_LABELS[c]:35s}: {w:.4f}")
 
def fbeta_score(y, preds, beta=2.0):
    tp = np.sum((preds == 1) & (y == 1))
    fp = np.sum((preds == 1) & (y == 0))
    fn = np.sum((preds == 0) & (y == 1))
    prec = tp / (tp + fp + EPS)
    rec  = tp / (tp + fn + EPS)
    return (1 + beta**2) * prec * rec / (beta**2 * prec + rec + EPS)
 
_BETA_THRESH_GRID = np.arange(0.05, 0.96, 0.01)
 
print(f"\n  ✓ Cost-sensitive F2 threshold grid ready ({len(_BETA_THRESH_GRID)} candidates)")
print(f"  ✓ _cached_splits defined ({len(_cached_splits)} folds)")
 
 

  Evaluation CV : 10-fold  (n=1520)
  Optuna CV     : 10-fold (now matches evaluation CV)
  Cached 10 fold splits for all OOF evaluations
══════════════════════════════════════════════════════════════════════════════
INDIVIDUAL MODEL PERFORMANCE  (B: 3-model (rPPG+Eff+Swin))
══════════════════════════════════════════════════════════════════════════════
Model                                   AUC     Acc      F1    Prec     Rec   Brier
──────────────────────────────────────────────────────────────────────────────
  rPPG (ML Stacking)                 0.6283  0.5993  0.6182  0.5686  0.6772  0.2383
  EfficientNet-B4 (BiLSTM+Attn)      0.7784  0.7283  0.7344  0.6904  0.7843  0.1919
  Swin-Tiny (BiLSTM+DCT)             0.7884  0.7237  0.7276  0.6892  0.7706  0.1876
══════════════════════════════════════════════════════════════════════════════
  Best individual: 0.7884 (Swin-Tiny (BiLSTM+DCT))

── Pairwise correlations (lower = more diverse = better ensemble) ──
    rPPG         ↔ EfficientNe

In [7]:
 
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 v5: ALL ENSEMBLE METHODS — MAXIMUM ACCURACY BUILD
# ════════════════════════════════════════════════════════════════════════════
#
# ★ CHANGE 5 — Replace Frank-Wolfe with SLSQP multi-start (30 restarts)
#   Reason: Frank-Wolfe is a gradient-free first-order method with slow
#   convergence on flat objectives. With 3 weights on a simplex, SLSQP
#   (Sequential Least Squares Programming) with random restarts finds the
#   global optimum reliably in ~30 × 500 = 15,000 calls — far fewer than
#   Frank-Wolfe's 200 × (M gradient estimates) = 600 objective calls, yet
#   with better convergence guarantees. In the original run Frank-Wolfe
#   converged to exactly uniform (1/3, 1/3, 1/3), which is a saddle point
#   on a flat AUC landscape, not the global maximum.
#
# ★ CHANGE 6 — Optuna: NSGAIISampler → TPESampler, 600 → 1500 trials,
#              add dedicated single-objective AUC study
#   Reason 1: For 3-parameter weight optimisation, TPE (Tree-structured
#   Parzen Estimator) converges ~3× faster than NSGA-II because it builds
#   a surrogate model of the objective. NSGA-II is designed for many
#   parameters (>10) where surrogate methods struggle.
#   Reason 2: 600 → 1500 trials gives TPE enough evaluations to fully
#   explore the 3-weight simplex.
#   Reason 3: We add a single-objective AUC study alongside the multi-
#   objective (AUC + log-loss) study and report the best of both.
#
# ★ CHANGE 7 — Meta-learner regularization improvements
#   Reason: All meta-learners underperformed simple average (a classic sign
#   of overfitting with n=1520 samples and ~19 features derived from 3
#   highly correlated base models). Increased regularization for each learner.
#     Meta-LR:   C=0.1  → C=0.01   (10× stronger L2 regularisation)
#     Meta-GBM:  lr=0.05 n=300 → lr=0.03 n=500, min_samples_leaf=15
#     Meta-RF:   no max_features → max_features='sqrt', min_samples_leaf=5
#     Meta-ET:   same changes as RF
#     Meta-LGB:  lr=0.05 → lr=0.03, min_child_samples=20
#     Meta-CAT:  lr=0.05 → lr=0.03, l2_leaf_reg=5
#     Meta-XGB:  lr=0.05 → lr=0.03, reg_lambda=2, min_child_weight=5
# ════════════════════════════════════════════════════════════════════════════
 
X_scores = merged[SCORE_COLS].values
ensemble_results = {}
 
def eval_ensemble(probs, name, yt, thresh=0.5):
    preds = (probs >= thresh).astype(int)
    auc   = roc_auc_score(yt, probs)
    acc   = accuracy_score(yt, preds)
    f1    = f1_score(yt, preds, zero_division=0)
    ensemble_results[name] = dict(auc=auc, acc=acc, f1=f1, probs=probs)
    return auc, acc, f1
 
def oof_predict(estimator, X, y, splits):
    oof = np.zeros(len(y))
    for tr, va in splits:
        estimator.fit(X[tr], y[tr])
        oof[va] = estimator.predict_proba(X[va])[:, 1]
    return oof
 
X_meta = build_meta_features_v3(X_scores)
 
print('═' * 70)
print(f'ENSEMBLE METHODS v5 (n={N}, {_cv_label}, {len(X_meta[0])}-dim meta-features)')
print('═' * 70)
print(f"{'Method':50s} {'AUC':>7} {'Acc':>7} {'F1':>7}")
print('─' * 70)
 
M   = X_scores.shape[1]
_N  = len(y_true)
splits = _cached_splits
 
# ─── CLOSED-FORM METHODS (unchanged) ─────────────────────────────────────────
P_hard = (X_scores >= 0.5).astype(float).mean(axis=1)
auc, acc, f1 = eval_ensemble(P_hard, 'hard_vote', y_true)
print(f"  {'Hard majority vote':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_simple = X_scores.mean(axis=1)
auc, acc, f1 = eval_ensemble(P_simple, 'simple_avg', y_true)
print(f"  {'Simple average':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
auc_w = np.array([individual_aucs[c] for c in SCORE_COLS]); auc_w /= auc_w.sum()
P_aucw = (X_scores * auc_w).sum(axis=1)
auc, acc, f1 = eval_ensemble(P_aucw, 'auc_weighted', y_true)
print(f"  {'AUC-weighted average':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_geo = np.exp(np.log(np.clip(X_scores, EPS, 1-EPS)).mean(axis=1))
auc, acc, f1 = eval_ensemble(np.clip(P_geo, 0, 1), 'geometric_mean', y_true)
print(f"  {'Geometric mean':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_logit_avg = expit(logit(np.clip(X_scores, EPS, 1-EPS)).mean(axis=1))
auc, acc, f1 = eval_ensemble(np.clip(P_logit_avg, 0, 1), 'logit_avg', y_true)
print(f"  {'Logit-space average (sigmoid(mean logit))':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_harm = M / np.sum(1.0 / np.clip(X_scores, EPS, 1-EPS), axis=1)
auc, acc, f1 = eval_ensemble(np.clip(P_harm, 0, 1), 'harmonic_mean', y_true)
print(f"  {'Harmonic mean':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
log_p   = np.log(np.clip(X_scores,   EPS, 1-EPS)).sum(axis=1)
log_1mp = np.log(np.clip(1-X_scores, EPS, 1-EPS)).sum(axis=1)
P_prod  = np.exp(log_p) / (np.exp(log_p) + np.exp(log_1mp) + EPS)
auc, acc, f1 = eval_ensemble(np.clip(P_prod, 0, 1), 'product_prob', y_true)
print(f"  {'Product of probabilities':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
conf_w     = np.abs(X_scores - 0.5)
conf_w_sum = conf_w.sum(axis=1, keepdims=True) + EPS
P_conf = (X_scores * conf_w / conf_w_sum).sum(axis=1)
auc, acc, f1 = eval_ensemble(np.clip(P_conf, 0, 1), 'conf_weighted_vote', y_true)
print(f"  {'Confidence-weighted vote (|p-0.5| weights)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
_trim = int(np.floor(M * 0.1))
P_trim = (np.sort(X_scores, axis=1)[:, _trim:M-_trim if _trim > 0 else None].mean(axis=1)
          if M > 2 else P_simple)
auc, acc, f1 = eval_ensemble(P_trim, 'trimmed_mean', y_true)
print(f"  {'Trimmed mean':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
auc_w_norm = auc_w / auc_w.sum()
def weighted_median_row(row, weights):
    idx = np.argsort(row)
    cs  = weights[idx].cumsum()
    return row[idx[np.searchsorted(cs, 0.5 * cs[-1])]]
P_wmed = np.array([weighted_median_row(X_scores[i], auc_w_norm) for i in range(_N)])
auc, acc, f1 = eval_ensemble(P_wmed, 'weighted_median', y_true)
print(f"  {'Weighted median (AUC weights)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_rank = np.column_stack([rankdata(X_scores[:, i]) / _N for i in range(M)]).mean(axis=1)
auc, acc, f1 = eval_ensemble(P_rank, 'rank_based', y_true)
print(f"  {'Rank-based average':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
ranks_b = np.array([rankdata(X_scores[:, i]) for i in range(M)])
P_borda = ranks_b.sum(axis=0) / (ranks_b.max() + EPS)
auc, acc, f1 = eval_ensemble(P_borda / P_borda.max(), 'borda_count', y_true)
print(f"  {'Borda count':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
P_ky = np.zeros(_N); cnt = 0
for i in range(M):
    for j in range(i+1, M):
        P_ky += (X_scores[:,i] + X_scores[:,j]) / 2; cnt += 1
P_ky /= max(cnt, 1)
auc, acc, f1 = eval_ensemble(P_ky, 'kemeny_young', y_true)
print(f"  {'Kemeny-Young':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
best_pm_auc, best_pm_p, best_pm_a = -1, None, 1.0
for alpha in [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]:
    Xc = np.clip(X_scores, EPS, 1-EPS)
    P_pm = (Xc.mean(axis=1) if abs(alpha - 1.0) < EPS
            else np.power(np.power(Xc, alpha).mean(axis=1), 1.0 / alpha))
    try:
        a = roc_auc_score(y_true, np.clip(P_pm, 0, 1))
        if a > best_pm_auc: best_pm_auc, best_pm_p, best_pm_a = a, P_pm, alpha
    except: pass
auc, acc, f1 = eval_ensemble(np.clip(best_pm_p, 0, 1), 'power_mean', y_true)
print(f"  {'Power mean (best α=' + str(best_pm_a) + ')':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
auc, acc, f1 = eval_ensemble(X_scores.min(axis=1), 'min_ensemble', y_true)
print(f"  {'Min ensemble (most conservative)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
auc, acc, f1 = eval_ensemble(X_scores.max(axis=1), 'max_ensemble', y_true)
print(f"  {'Max ensemble (most aggressive)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# ─── LEARNED METHODS — ★ CHANGE 7: stronger regularisation ──────────────────
 
# Meta-LR  ★ C=0.1 → C=0.01
lr_meta = Pipeline([('sc', StandardScaler()),
                    ('lr', LogisticRegression(C=0.01, max_iter=1000,   # ★ was C=0.1
                                              class_weight='balanced',
                                              random_state=SEED))])
P_lr = oof_predict(lr_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_lr, 'meta_lr', y_true)
print(f"  {'Meta-LR (enriched features, OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-MLP  ★ smaller network, higher alpha
mlp_meta = Pipeline([('sc', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',  # ★ was (128,64,32)
                          alpha=0.05, max_iter=500, random_state=SEED,     # ★ alpha 0.01→0.05
                          early_stopping=True, validation_fraction=0.15,
                          n_iter_no_change=15))])
P_mlp = oof_predict(mlp_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_mlp, 'meta_mlp', y_true)
print(f"  {'Meta-MLP neural (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-SVC  (unchanged — already calibrated)
svc_meta = Pipeline([('sc', StandardScaler()),
    ('svc', CalibratedClassifierCV(
        SVC(kernel='rbf', C=1.0, probability=False, random_state=SEED),
        method='sigmoid', cv=3))])
P_svc = oof_predict(svc_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_svc, 'meta_svc', y_true)
print(f"  {'Meta-SVC RBF (calibrated, OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-GBM  ★ lr=0.05→0.03, n=300→500, min_samples_leaf 1→15
gbm_meta = GradientBoostingClassifier(
    n_estimators=500, max_depth=3,           # ★ was n=300
    learning_rate=0.03,                       # ★ was 0.05
    subsample=0.8, min_samples_leaf=15,       # ★ added min_samples_leaf=15
    random_state=SEED)
P_gbm = oof_predict(gbm_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_gbm, 'meta_gbm', y_true)
print(f"  {'Meta-GBM (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-RF  ★ max_features='sqrt', min_samples_leaf=5
rf_meta = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    min_samples_leaf=5,                       # ★ was 2
    max_features='sqrt',                      # ★ added
    class_weight='balanced', random_state=SEED, n_jobs=-1)
P_rf = oof_predict(rf_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_rf, 'meta_rf', y_true)
print(f"  {'Meta-RF (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-ET  ★ same as RF
et_meta = ExtraTreesClassifier(
    n_estimators=300, max_depth=None,
    min_samples_leaf=5,                       # ★ was 2
    max_features='sqrt',                      # ★ added
    class_weight='balanced', random_state=SEED, n_jobs=-1)
P_et = oof_predict(et_meta, X_meta, y_true, splits)
auc, acc, f1 = eval_ensemble(P_et, 'meta_et', y_true)
print(f"  {'Meta-ExtraTrees (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-LGB  ★ lr=0.05→0.03, min_child_samples 1→20
if HAS_LGB:
    lgb_meta = lgb.LGBMClassifier(
        n_estimators=500, max_depth=4,
        learning_rate=0.03,                   # ★ was 0.05
        subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20,                 # ★ added — prevents overfitting leaves
        class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)
    P_lgb = oof_predict(lgb_meta, X_meta, y_true, splits)
    auc, acc, f1 = eval_ensemble(P_lgb, 'meta_lgbm', y_true)
    print(f"  {'Meta-LightGBM (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-CAT  ★ lr=0.05→0.03, l2_leaf_reg=5
if HAS_CAT:
    cat_meta = CatBoostClassifier(
        iterations=500, depth=4,
        learning_rate=0.03,                   # ★ was 0.05
        l2_leaf_reg=5,                        # ★ added L2 regularisation
        auto_class_weights='Balanced',
        random_seed=SEED, verbose=0)
    P_cat = oof_predict(cat_meta, X_meta, y_true, splits)
    auc, acc, f1 = eval_ensemble(P_cat, 'meta_catboost', y_true)
    print(f"  {'Meta-CatBoost (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Meta-XGB  ★ lr=0.05→0.03, reg_lambda=2, min_child_weight=5
if HAS_XGB:
    xgb_meta = XGBClassifier(
        n_estimators=500, max_depth=3,
        learning_rate=0.03,                   # ★ was 0.05
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0,                       # ★ added L2
        min_child_weight=5,                   # ★ added — prevents tiny leaves
        eval_metric='logloss',
        random_state=SEED, n_jobs=-1, verbosity=0)
    P_xgb = oof_predict(xgb_meta, X_meta, y_true, splits)
    auc, acc, f1 = eval_ensemble(P_xgb, 'meta_xgb', y_true)
    print(f"  {'Meta-XGBoost (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# Mixture-of-Experts (unchanged — bug was already fixed in v4)
def mixture_of_experts_oof(X, y, spl):
    oof = np.zeros(len(y))
    nm  = X.shape[1]
    for tr, va in spl:
        X_tr, y_tr = X[tr], y[tr]
        gate_scores = np.zeros((len(va), nm))
        for m_i in range(nm):
            correct_tr = ((X_tr[:, m_i] >= 0.5).astype(int) == y_tr).astype(float)
            if len(np.unique(correct_tr)) < 2:
                gate_scores[:, m_i] = float(correct_tr.mean()); continue
            gater = LogisticRegression(C=0.5, max_iter=300, random_state=SEED)
            try:
                gater.fit(X_tr, correct_tr)
                gate_scores[:, m_i] = gater.predict_proba(X[va])[:, 1]
            except Exception:
                gate_scores[:, m_i] = float(correct_tr.mean())
        gs    = gate_scores - gate_scores.max(axis=1, keepdims=True)
        gates = np.exp(gs)
        gates = gates / (gates.sum(axis=1, keepdims=True) + EPS)
        oof[va] = (X[va] * gates).sum(axis=1)
    return oof
 
P_moe = mixture_of_experts_oof(X_scores, y_true, splits)
auc, acc, f1 = eval_ensemble(P_moe, 'mixture_of_experts', y_true)
print(f"  {'Mixture-of-Experts gating (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# DES-KNN (unchanged)
def des_knn_oof(X, y, spl, k=7):
    from sklearn.neighbors import NearestNeighbors
    oof = np.zeros(len(y))
    for tr, va in spl:
        X_tr, y_tr, X_va = X[tr], y[tr], X[va]
        nn = NearestNeighbors(n_neighbors=min(k, len(tr)), algorithm='ball_tree')
        nn.fit(X_tr)
        _, idx = nn.kneighbors(X_va)
        for s_i, (vi, neigh_idx) in enumerate(zip(va, idx)):
            neigh_y = y_tr[neigh_idx]
            if len(np.unique(neigh_y)) < 2:
                oof[vi] = X[vi].mean(); continue
            best_m, best_acc = 0, -1
            for m_i in range(X.shape[1]):
                neigh_pred = (X_tr[neigh_idx, m_i] > 0.5).astype(int)
                acc_m = accuracy_score(neigh_y, neigh_pred)
                if acc_m > best_acc: best_acc, best_m = acc_m, m_i
            oof[vi] = X[vi, best_m]
    return oof
 
P_des = des_knn_oof(X_scores, y_true, splits, k=7)
auc, acc, f1 = eval_ensemble(P_des, 'des_knn', y_true)
print(f"  {'Dynamic Ensemble Selection KNN (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
# ─── ★ CHANGE 5: SLSQP MULTI-START (replaces Frank-Wolfe) ────────────────────
print(f"\n  SLSQP multi-start weight optimisation (30 restarts, simplex-constrained)...")




def slsqp_weights(Xs, y, spl, n_restarts=30):
    """
    Multi-start SLSQP on probability simplex.
    Replaces Frank-Wolfe which converged to uniform (1/3,1/3,1/3) in v4.
    """
    M = Xs.shape[1]
    best_auc, best_w = -1.0, np.ones(M) / M

    def neg_oof_auc(w):
        w_n = np.clip(w, 0, 1)
        s   = w_n.sum()
        if s < EPS: return 1.0          # ★ FIX: was 0.0 — 1.0 = max penalty (AUC 0)
        w_n /= s
        oof = np.zeros(len(y))
        for tr, va in spl:
            oof[va] = Xs[va] @ w_n
        try:
            return -roc_auc_score(y, oof)
        except Exception:
            return 1.0                   # ★ FIX: was 0.0

    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
    bounds      = [(0.0, 1.0)] * M

    for seed_ in range(n_restarts):
        rng = np.random.RandomState(seed_)
        w0  = rng.dirichlet(np.ones(M))
        try:
            res = minimize(neg_oof_auc, w0, method='SLSQP',
                           constraints=constraints, bounds=bounds,
                           options={'maxiter': 1000, 'ftol': 1e-10})
            w_sol = np.clip(res.x, 0, 1)
            w_sol /= w_sol.sum() + EPS
            auc_sol = -res.fun
            if auc_sol > best_auc:
                best_auc = auc_sol
                best_w   = w_sol
        except Exception:
            pass

    best_w = np.clip(best_w, 0, 1)
    best_w /= best_w.sum()
    return best_w, best_auc

slsqp_w, slsqp_auc = slsqp_weights(X_scores, y_true, splits, n_restarts=30)
fw_w = slsqp_w   # keep fw_w name for backward compat with artifact key
P_fw = np.zeros(_N)
for tr, va in splits:
    P_fw[va] = X_scores[va] @ fw_w
auc, acc, f1 = eval_ensemble(P_fw, 'frank_wolfe_weights', y_true)
print(f"  {'SLSQP multi-start weights (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
for i, c in enumerate(SCORE_COLS):
    print(f"    {MODEL_LABELS[c]}: {fw_w[i]:.4f}")









# ─── ★ CHANGE 6: OPTUNA — TPE + single-objective + 1500 trials ───────────────
print(f"\n  Optuna weight optimisation (TPE, 1500 trials, {_cv_label})...")
_swa_buffer = []
 
def _oof_auc_ll(w, Xs, y, spl):
    """Compute OOF AUC and log-loss for given weights."""
    w_n = np.clip(w, 0, 1); w_n /= w_n.sum() + EPS
    oof = np.zeros(len(y))
    for tr, va in spl:
        oof[va] = Xs[va] @ w_n
    oof = np.clip(oof, EPS, 1-EPS)
    return roc_auc_score(y, oof), log_loss(y, oof)   # ★ log-loss replaces Brier
 
if HAS_OPTUNA:
    # ── Single-objective: maximise OOF AUC ──────────────────────────────────
    def _single_obj(trial):
        w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(M)])
        if w.sum() < EPS: return 0.5
        auc_, ll_ = _oof_auc_ll(w, X_scores, y_true, splits)
        _swa_buffer.append((auc_, (w / w.sum()).copy()))
        return -auc_   # minimise
 
    study_single = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=SEED))    # ★ TPE replaces NSGA-II
    study_single.optimize(_single_obj, n_trials=1500,     # ★ 600 → 1500
                           show_progress_bar=False)
    best_w_single = np.array([study_single.best_params[f'w{i}'] for i in range(M)])
    best_w_single /= best_w_single.sum()
 
    # ── Multi-objective: (AUC, log-loss) for Pareto front ────────────────────
    def _multi_obj(trial):
        w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(M)])
        if w.sum() < EPS: return 0.0, 10.0
        auc_, ll_ = _oof_auc_ll(w, X_scores, y_true, splits)
        _swa_buffer.append((auc_, (w / w.sum()).copy()))
        return -auc_, ll_
 
    study_multi = optuna.create_study(
        directions=['minimize', 'minimize'],
        sampler=optuna.samplers.NSGAIISampler(seed=SEED+1))
    study_multi.optimize(_multi_obj, n_trials=600,
                          show_progress_bar=False)
    pareto = study_multi.best_trials
    best_trial_multi = min(pareto, key=lambda t: t.values[0])
    best_w_multi = np.array([best_trial_multi.params[f'w{i}'] for i in range(M)])
    best_w_multi /= best_w_multi.sum()
 
    # ── Pick the better of single vs multi-objective ─────────────────────────
    oof_s = np.zeros(_N)
    for tr, va in splits: oof_s[va] = X_scores[va] @ best_w_single
    oof_m = np.zeros(_N)
    for tr, va in splits: oof_m[va] = X_scores[va] @ best_w_multi
    auc_s = roc_auc_score(y_true, oof_s)
    auc_m = roc_auc_score(y_true, oof_m)
 
    if auc_s >= auc_m:
        best_w_opt = best_w_single
        print(f"  ★ TPE single-objective wins (AUC={auc_s:.4f} vs multi={auc_m:.4f})")
    else:
        best_w_opt = best_w_multi
        print(f"  ★ NSGA-II multi-objective wins (AUC={auc_m:.4f} vs single={auc_s:.4f})")
else:
    best_w_opt = fw_w   # fallback to SLSQP weights
 
P_opt = np.zeros(_N)
for tr, va in splits: P_opt[va] = X_scores[va] @ best_w_opt
auc, acc, f1 = eval_ensemble(P_opt, 'optuna_multi_obj', y_true)
print(f"  {'Optuna optimised weights (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
for i, c in enumerate(SCORE_COLS):
    print(f"    {MODEL_LABELS[c]}: {best_w_opt[i]:.4f}")
 
# SWA
if _swa_buffer:
    _swa_buffer.sort(key=lambda x: -x[0])
    top_k = max(20, len(_swa_buffer) // 10)
    swa_w = np.average([w for _, w in _swa_buffer[:top_k]],
                        weights=[a for a, _ in _swa_buffer[:top_k]], axis=0)
    swa_w /= swa_w.sum()
    P_swa = X_scores @ swa_w
    auc, acc, f1 = eval_ensemble(P_swa, 'swa_weights', y_true)
    print(f"  {'SWA (AUC-weighted top configs)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
else:
    ensemble_results['swa_weights'] = ensemble_results['optuna_multi_obj']
 
# Super-ensemble L2 (unchanged)
print(f"\n  Level-2 super-ensemble (stack of all meta-learner OOF outputs)...")
_LEARNED_KEYS = ['meta_lr','meta_mlp','meta_svc','meta_gbm','meta_rf','meta_et',
                  'des_knn','optuna_multi_obj','frank_wolfe_weights','swa_weights']
if HAS_LGB and 'meta_lgbm'     in ensemble_results: _LEARNED_KEYS.append('meta_lgbm')
if HAS_CAT and 'meta_catboost' in ensemble_results: _LEARNED_KEYS.append('meta_catboost')
if HAS_XGB and 'meta_xgb'      in ensemble_results: _LEARNED_KEYS.append('meta_xgb')
 
available_keys = [k for k in _LEARNED_KEYS if k in ensemble_results]
if len(available_keys) >= 3:
    X_super = np.column_stack([ensemble_results[k]['probs'] for k in available_keys])
    closed_extra = np.column_stack([
        np.clip(P_geo, 0, 1), np.clip(P_logit_avg, 0, 1),
        P_aucw, np.clip(P_prod, 0, 1)
    ])
    X_super = np.hstack([X_super, closed_extra])
    super_lr = Pipeline([('sc', StandardScaler()),
                          ('lr', LogisticRegression(C=0.05, max_iter=500,
                                                    class_weight='balanced',
                                                    random_state=SEED))])
    P_super = oof_predict(super_lr, X_super, y_true, splits)
    auc, acc, f1 = eval_ensemble(P_super, 'super_ensemble_L2', y_true)
    print(f"  {'Super-ensemble Level-2 (OOF)':48s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
 
print('\n' + '═' * 70)
print(f"  ✓ {len(ensemble_results)} ensemble methods evaluated.")
print('═' * 70)
 
# Strategy comparison (unchanged)
print('\n' + '═' * 70)
print('STRATEGY × METHOD COMPARISON')
print('═' * 70)
print(f"  {'Strategy':32s} {'n':>4} {'Simple':>8} {'AUCWtd':>8} {'GeoMean':>8}")
print("  " + "─" * 66)
for sname, (smdf, scols) in all_strategies.items():
    syt = smdf['label'].values; sXs = smdf[scols].values; n = len(smdf)
    try:
        a_s = roc_auc_score(syt, sXs.mean(axis=1))
        ww  = np.array([roc_auc_score(syt, smdf[c].values) for c in scols])
        ww /= ww.sum()
        a_w = roc_auc_score(syt, (sXs * ww).sum(axis=1))
        a_g = roc_auc_score(syt,
              np.clip(np.exp(np.log(np.clip(sXs, EPS, 1-EPS)).mean(axis=1)), 0, 1))
        print(f"  {sname:32s} {n:>4} {a_s:>8.4f} {a_w:>8.4f} {a_g:>8.4f}")
    except Exception as e:
        print(f"  {sname:32s} {n:>4}  ERROR: {e}")
print("  " + "─" * 66)
 
 

══════════════════════════════════════════════════════════════════════
ENSEMBLE METHODS v5 (n=1520, 10-fold, 22-dim meta-features)
══════════════════════════════════════════════════════════════════════
Method                                                 AUC     Acc      F1
──────────────────────────────────────────────────────────────────────
  Hard majority vote                                0.7556  0.7230  0.7122
  Simple average                                    0.8022  0.7263  0.7151
  AUC-weighted average                              0.8044  0.7237  0.7127
  Geometric mean                                    0.8006  0.7243  0.7060
  Logit-space average (sigmoid(mean logit))         0.8016  0.7276  0.7164
  Harmonic mean                                     0.7959  0.7263  0.7020
  Product of probabilities                          0.8016  0.7276  0.7164
  Confidence-weighted vote (|p-0.5| weights)        0.7978  0.7250  0.7125
  Trimmed mean                                      

In [8]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 v5: SELECT BEST + TEMPERATURE SCALING + DELONG + VENN-ABERS
# ════════════════════════════════════════════════════════════════════════════
#
# ★ CHANGE 8 — Fix temperature scaling log-loss discrepancy
#   Root cause (v4 bug): The OOF log-loss (0.5358) was computed with per-fold
#   Nelder-Mead-optimised T values (T_fold), but then `best_T` was set to
#   T_CAND (the grid search starting point), not T_fold. Applying T_cand
#   globally gave 0.6364 — worse than the OOF estimate. Fix: store the MEAN
#   of per-fold optimised T values for the best T_cand, then apply that mean T
#   globally. This makes the global application consistent with OOF estimate.
#
# ★ CHANGE 9 — Extended temperature grid: [0.3, 4.0] 50 steps → [0.05, 8.0] 100 steps
#   Reason: In v4, T=3.924 hit near the top of the [0.3, 4.0] grid, suggesting
#   the true optimal T may be higher. Extending to 8.0 and doubling resolution
#   finds the true minimum. Also extending downward to 0.05 allows sharpening
#   under-confident models.
# ════════════════════════════════════════════════════════════════════════════
 
_OOF_KEYS    = {'meta_lr','meta_mlp','meta_svc','meta_gbm','meta_rf','meta_et',
                'meta_lgbm','meta_catboost','meta_xgb','optuna_multi_obj',
                'frank_wolfe_weights','swa_weights','mixture_of_experts',
                'des_knn','super_ensemble_L2'}
_CLOSED_KEYS = {'hard_vote','simple_avg','auc_weighted','geometric_mean',
                'logit_avg','harmonic_mean','product_prob','conf_weighted_vote',
                'trimmed_mean','weighted_median','rank_based','borda_count',
                'kemeny_young','power_mean','min_ensemble','max_ensemble'}
 
oof_cands    = {k: v for k, v in ensemble_results.items() if k in _OOF_KEYS}
closed_cands = {k: v for k, v in ensemble_results.items() if k in _CLOSED_KEYS}
best_oof     = max(oof_cands,    key=lambda k: oof_cands[k]['auc'])    if oof_cands    else None
best_closed  = max(closed_cands, key=lambda k: closed_cands[k]['auc']) if closed_cands else None
 
OOF_TOLERANCE = 0.002
if best_oof and best_closed:
    auc_oof    = ensemble_results[best_oof]['auc']
    auc_closed = ensemble_results[best_closed]['auc']
    best_name  = best_oof if auc_oof >= auc_closed - OOF_TOLERANCE else best_closed
    print(f"  Best OOF     : {best_oof} (AUC={auc_oof:.4f})")
    print(f"  Best closed  : {best_closed} (AUC={auc_closed:.4f})")
elif best_oof:
    best_name = best_oof
elif best_closed:
    best_name = best_closed
else:
    best_name = max(ensemble_results, key=lambda k: ensemble_results[k]['auc'])
 
best_probs_raw = ensemble_results[best_name]['probs']
 
# ─── ★ CHANGE 8+9: Fixed temperature scaling ─────────────────────────────────
def temperature_scale_oof(probs, y, spl):
    """
    ★ FIX v5 (two fixes):
    1. Extended grid: [0.05, 8.0] with 100 steps (was [0.3, 4.0] with 50).
       Prevents grid-boundary saturation (v4 T=3.924 was near the 4.0 ceiling).
    2. best_T = mean of per-fold optimised T values (not T_cand).
       In v4, best_T was set to T_cand (the grid point), but OOF log-loss was
       computed with per-fold T_fold (Nelder-Mead optimised). This mismatch
       caused global log-loss (0.6364) >> OOF estimate (0.5358). Storing the
       mean per-fold T makes the global application consistent with the OOF
       estimate. Expected gap after fix: < 0.01.
    """
    EPS_   = 1e-7
    best_T = 1.0
    best_ll = log_loss(y, np.clip(probs, EPS_, 1-EPS_))
    best_fold_Ts = [1.0] * len(spl)
 
    for T_cand in np.linspace(0.05, 8.0, 100):  # ★ extended grid
        oof_t    = np.zeros(len(y))
        fold_Ts  = []
        ok       = True
        for tr, va in spl:
            def _ll_fold(t, _tr=tr):
                sc = expit(logit(np.clip(probs[_tr], EPS_, 1-EPS_)) / max(0.01, t[0]))
                return log_loss(y[_tr], sc)
            try:
                res = minimize(_ll_fold, x0=[T_cand], method='Nelder-Mead',
                               options={'maxiter': 80, 'xatol': 1e-3, 'fatol': 1e-4})
                T_fold = float(np.clip(res.x[0], 0.05, 12.0))
            except Exception:
                T_fold = T_cand
            fold_Ts.append(T_fold)
            oof_t[va] = expit(logit(np.clip(probs[va], EPS_, 1-EPS_)) / T_fold)
        try:
            ll = log_loss(y, np.clip(oof_t, EPS_, 1-EPS_))
            if ll < best_ll:
                best_ll      = ll
                best_T       = float(np.mean(fold_Ts))  # ★ mean per-fold T (not T_cand)
                best_fold_Ts = fold_Ts[:]
        except Exception:
            pass
 
    return best_T, best_ll
 
print("  Running OOF temperature scaling...")
best_T, T_ll = temperature_scale_oof(best_probs_raw, y_true, splits)
raw_ll  = log_loss(y_true, np.clip(best_probs_raw, EPS, 1-EPS))
raw_auc = roc_auc_score(y_true, best_probs_raw)
 
# ★ CHANGE 8: Apply the mean-fold T globally and compute correct final log-loss
scaled_probs = expit(logit(np.clip(best_probs_raw, EPS, 1-EPS)) / best_T)
scaled_ll    = log_loss(y_true, np.clip(scaled_probs, EPS, 1-EPS))
scaled_auc   = roc_auc_score(y_true, scaled_probs)
 
if T_ll < raw_ll:
    best_probs = scaled_probs
    final_name = best_name + f'+T{best_T:.2f}'
    print(f"  OOF temp T={best_T:.3f} | log-loss {raw_ll:.4f} → OOF={T_ll:.4f}, "
          f"global={scaled_ll:.4f} ✓")
else:
    best_probs = best_probs_raw
    final_name = best_name
    best_T     = 1.0
    print(f"  OOF temp T={best_T:.3f} | no improvement (log-loss={raw_ll:.4f}) — keeping raw")
print(f"  AUC unchanged: {roc_auc_score(y_true, best_probs):.4f}")
 
# Venn-Abers (unchanged)
def venn_abers_oof(probs, y, spl):
    oof = np.zeros(len(y))
    for tr, va in spl:
        p_tr, y_tr = probs[tr], y[tr]
        sort_idx   = np.argsort(p_tr)
        p_sorted, y_sorted = p_tr[sort_idx], y_tr[sort_idx]
        for i, p_test in enumerate(probs[va]):
            def _iso(p_aug, y_aug, _pt=p_test):
                from sklearn.isotonic import IsotonicRegression as IR
                return IR(out_of_bounds='clip').fit(p_aug, y_aug).predict([_pt])[0]
            p0 = _iso(np.append(p_sorted, p_test), np.append(y_sorted, 0))
            p1 = _iso(np.append(p_sorted, p_test), np.append(y_sorted, 1))
            oof[va[i]] = (p0 + p1) / 2.0
    return np.clip(oof, 0.0, 1.0)
 
print("  Applying Venn-Abers calibration...")
P_va   = venn_abers_oof(best_probs, y_true, splits)
auc_va = roc_auc_score(y_true, P_va)
ensemble_results['venn_abers_final'] = dict(auc=auc_va, acc=0, f1=0, probs=P_va)
if auc_va > roc_auc_score(y_true, best_probs):
    best_probs = P_va; final_name += '+VennAbers'
    print(f"  ✓ Venn-Abers improved AUC to {auc_va:.4f}")
else:
    print(f"  Venn-Abers AUC={auc_va:.4f} (no improvement — keeping)")
 
# BMA candidate (unchanged)
X_sc  = merged[SCORE_COLS].values
P_bma = (X_sc * bma_weights[None, :]).sum(axis=1)
bma_auc = roc_auc_score(y_true, P_bma)
ensemble_results['bma_weights'] = dict(auc=bma_auc, acc=0, f1=0, probs=P_bma)
print(f"\n  BMA ensemble: AUC={bma_auc:.4f}")
if bma_auc > roc_auc_score(y_true, best_probs):
    best_probs = P_bma; final_name += '+BMA'
    print(f"  ✓ BMA improved best ensemble AUC to {bma_auc:.4f}")
 
# DeLong test (unchanged — bug fix was already in v4)
def delong_test(y, pred_a, pred_b):
    def _auc_var(y, pred):
        pos = pred[y==1]; neg = pred[y==0]
        n1, n0 = len(pos), len(neg)
        psi_pos = np.array([np.mean(p>neg)+0.5*np.mean(p==neg) for p in pos])
        psi_neg = np.array([np.mean(q<pos)+0.5*np.mean(q==pos) for q in neg])
        var_pos = float(np.var(psi_pos, ddof=1)) if n1 > 1 else 0.0
        var_neg = float(np.var(psi_neg, ddof=1)) if n0 > 1 else 0.0
        return roc_auc_score(y, pred), var_pos / n1 + var_neg / n0
    auc_a, var_a = _auc_var(y, pred_a)
    auc_b, var_b = _auc_var(y, pred_b)
    se = np.sqrt(max(var_a + var_b, EPS))
    z  = (auc_a - auc_b) / se
    p  = 2 * (1 - norm.cdf(abs(z)))
    return z, p, auc_a, auc_b
 
best_ind = max(individual_aucs, key=individual_aucs.get)
z_dl, p_dl, auc_ens, auc_ind = delong_test(y_true, best_probs, merged[best_ind].values)
print(f"\n  DeLong: Ensemble ({auc_ens:.4f}) vs {MODEL_LABELS[best_ind]} ({auc_ind:.4f})")
print(f"    z={z_dl:.3f}  p={p_dl:.4f}  "
      f"{'★ Significant (p<0.05)' if p_dl < 0.05 else 'Not significant'}")
 
# McNemar (unchanged)
fpr_arr, tpr_arr, thresh_arr = roc_curve(y_true, best_probs)
best_thresh = float(thresh_arr[np.argmax(tpr_arr - fpr_arr)])
final_preds = (best_probs >= best_thresh).astype(int)
 
fpr_i, tpr_i, thr_i = roc_curve(y_true, merged[best_ind].values)
thresh_ind  = float(thr_i[np.argmax(tpr_i - fpr_i)])
preds_ind   = (merged[best_ind].values >= thresh_ind).astype(int)
b = np.sum((final_preds==y_true) & (preds_ind!=y_true))
c = np.sum((final_preds!=y_true) & (preds_ind==y_true))
mc_stat = (abs(b-c)-1)**2 / (b+c) if (b+c) > 0 else 0.0
mc_p    = 1 - chi2.cdf(mc_stat, df=1)
print(f"  McNemar: b={b} c={c}  χ²={mc_stat:.3f}  p={mc_p:.4f}  "
      f"{'★ Significant' if mc_p < 0.05 else 'Not significant'}")
 
# F2 threshold
best_f2_thresh, best_f2 = 0.5, 0.0
for t in _BETA_THRESH_GRID:
    f2 = fbeta_score(y_true, (best_probs >= t).astype(int), beta=2.0)
    if f2 > best_f2: best_f2, best_f2_thresh = f2, t
 
# Conformal prediction (unchanged)
def conformal_prediction(probs, y, alpha=0.1):
    cal_idx = _cached_splits[-1][1]
    cal_scores, cal_labels = probs[cal_idx], y[cal_idx]
    nc_scores = np.where(cal_labels==1, 1-cal_scores, cal_scores)
    q_level   = min(np.ceil((1-alpha)*(len(cal_idx)+1))/len(cal_idx), 1.0)
    threshold = np.quantile(nc_scores, q_level)
    include_fake = (1-probs) <= threshold
    include_real = probs     <= threshold
    labels_out = []
    for ir, ifk in zip(include_real, include_fake):
        if ir and ifk:  labels_out.append('Uncertain')
        elif ifk:       labels_out.append('Fake')
        elif ir:        labels_out.append('Real')
        else:           labels_out.append('Empty')
    labels_arr  = np.array(labels_out)
    n_uncertain = (labels_arr=='Uncertain').sum()
    coverage    = np.mean([l==('Fake' if y[i]==1 else 'Real') or l=='Uncertain'
                           for i, l in enumerate(labels_arr)])
    print(f"\n  Conformal prediction (α={alpha}):")
    print(f"    Coverage : {coverage:.4f}  (guaranteed ≥ {1-alpha:.2f})")
    print(f"    Uncertain: {n_uncertain}/{len(y)}  ({100*n_uncertain/len(y):.1f}%)")
    return labels_arr, threshold
 
conformal_labels, conformal_threshold = conformal_prediction(best_probs, y_true, alpha=0.1)
 
# ─── FINAL METRICS — ★ CHANGE 8: all computed from temperature-scaled best_probs
final_auc  = roc_auc_score(y_true, best_probs)
final_acc  = accuracy_score(y_true, final_preds)
final_f1   = f1_score(y_true, final_preds, zero_division=0)
final_prec = precision_score(y_true, final_preds, zero_division=0)
final_rec  = recall_score(y_true, final_preds, zero_division=0)
final_ap   = average_precision_score(y_true, best_probs)
final_ll   = log_loss(y_true, np.clip(best_probs, EPS, 1-EPS))   # ★ from scaled probs
 
print('\n' + '═'*70)
print(f'BEST ENSEMBLE v5: {final_name.upper()}')
print('═'*70)
print(f"  Threshold (Youden) : {best_thresh:.4f}")
print(f"  Threshold (F2)     : {best_f2_thresh:.4f}  ← use this if missing fakes is costly")
print(f"  AUC-ROC  : {final_auc:.4f}")
print(f"  Avg Prec : {final_ap:.4f}")
print(f"  Log-Loss : {final_ll:.4f}")
print(f"  Accuracy : {final_acc:.4f}")
print(f"  F1-Score : {final_f1:.4f}")
print(f"  Precision: {final_prec:.4f}")
print(f"  Recall   : {final_rec:.4f}")
print('═'*70)
print()
print(classification_report(y_true, final_preds,
                              target_names=['Real','Fake'], digits=4))
 
cv_thresholds = []
for tr_idx, val_idx in splits:
    val_p, val_y = best_probs[val_idx], y_true[val_idx]
    best_cv_f1, best_cv_t = 0.0, 0.5
    for t in np.arange(0.05, 0.96, 0.01):
        f1t = f1_score(val_y, (val_p>=t).astype(int), zero_division=0)
        if f1t > best_cv_f1: best_cv_f1, best_cv_t = f1t, t
    cv_thresholds.append(best_cv_t)
cv_thresh = float(np.mean(cv_thresholds))
cv_preds  = (best_probs >= cv_thresh).astype(int)
print(f"  ★ PRIMARY threshold  : {best_thresh:.4f} (Youden's J)")
print(f"    F2-optimal         : {best_f2_thresh:.4f} (recall-biased)")
print(f"    CV F1-optimal      : {cv_thresh:.4f} "
      f"(F1={f1_score(y_true, cv_preds, zero_division=0):.4f})")
 

  Best OOF     : optuna_multi_obj (AUC=0.8081)
  Best closed  : auc_weighted (AUC=0.8044)
  Running OOF temperature scaling...
  OOF temp T=0.711 | log-loss 0.5462 → OOF=0.5361, global=0.5357 ✓
  AUC unchanged: 0.8081
  Applying Venn-Abers calibration...
  Venn-Abers AUC=0.8000 (no improvement — keeping)

  BMA ensemble: AUC=0.8034

  DeLong: Ensemble (0.8081) vs Swin-Tiny (BiLSTM+DCT) (0.7884)
    z=1.235  p=0.2168  Not significant
  McNemar: b=47 c=39  χ²=0.570  p=0.4504  Not significant

  Conformal prediction (α=0.1):
    Coverage : 0.9007  (guaranteed ≥ 0.90)
    Uncertain: 613/1520  (40.3%)

══════════════════════════════════════════════════════════════════════
BEST ENSEMBLE v5: OPTUNA_MULTI_OBJ+T0.71
══════════════════════════════════════════════════════════════════════
  Threshold (Youden) : 0.4377
  Threshold (F2)     : 0.1600  ← use this if missing fakes is costly
  AUC-ROC  : 0.8081
  Avg Prec : 0.7869
  Log-Loss : 0.5357
  Accuracy : 0.7289
  F1-Score : 0.7335
  Precision: 

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 v3: BOOTSTRAP CIs + JACKKNIFE CI + DELONG CI
# v3 adds: jackknife-after-bootstrap CI, DeLong analytical CI stored to CSV
# ═══════════════════════════════════════════════════════════════════════════════

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, ci=0.95, seed=42):
    rng = np.random.RandomState(seed)
    classes = np.unique(y_true)
    idx_by_class = {c: np.where(y_true == c)[0] for c in classes}
    scores = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(idx_by_class[c], len(idx_by_class[c]), replace=True)
                               for c in classes])
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2: continue
        try: scores.append(metric_fn(yt, yp))
        except: continue
    if not scores: return 0.0, 0.0, 0.0
    alpha = (1 - ci) / 2
    return float(np.mean(scores)), float(np.percentile(scores, alpha*100)), float(np.percentile(scores, (1-alpha)*100))

def jackknife_ci(y_true, y_pred, metric_fn):
    """Jackknife (leave-one-out) CI — fast analytical alternative."""
    n = len(y_true)
    jk_vals = []
    base = metric_fn(y_true, y_pred)
    for i in range(n):
        idx = np.concatenate([np.arange(0, i), np.arange(i+1, n)])
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2: jk_vals.append(base); continue
        try: jk_vals.append(metric_fn(yt, yp))
        except: jk_vals.append(base)
    jk_arr = np.array(jk_vals)
    jk_mean = jk_arr.mean()
    jk_se   = np.sqrt((n - 1) * jk_arr.var())
    return jk_mean, jk_mean - 1.96 * jk_se, jk_mean + 1.96 * jk_se

print('Computing 95% Bootstrap CIs (n=2000, stratified) + Jackknife CIs...')

auc_m, auc_lo, auc_hi = bootstrap_ci(y_true, best_probs,  roc_auc_score)
acc_m, acc_lo, acc_hi = bootstrap_ci(y_true, final_preds, accuracy_score)
f1_m,  f1_lo,  f1_hi  = bootstrap_ci(y_true, final_preds, lambda t,p: f1_score(t, p, zero_division=0))
pre_m, pre_lo, pre_hi = bootstrap_ci(y_true, final_preds, lambda t,p: precision_score(t, p, zero_division=0))
rec_m, rec_lo, rec_hi = bootstrap_ci(y_true, final_preds, lambda t,p: recall_score(t, p, zero_division=0))

# Jackknife AUC CI
jk_auc_m, jk_auc_lo, jk_auc_hi = jackknife_ci(y_true, best_probs, roc_auc_score)

print('═' * 65)
print('FINAL ENSEMBLE v3 — CONFIDENCE INTERVALS')
print('═' * 65)
print(f'  Method: stratified bootstrap (n=2000) + jackknife')
print()
print(f'  AUC-ROC   bootstrap: {auc_m:.4f}  [{auc_lo:.4f}, {auc_hi:.4f}]  w={auc_hi-auc_lo:.4f}')
print(f'  AUC-ROC   jackknife: {jk_auc_m:.4f}  [{jk_auc_lo:.4f}, {jk_auc_hi:.4f}]')
print(f'  Accuracy           : {acc_m:.4f}  [{acc_lo:.4f}, {acc_hi:.4f}]  w={acc_hi-acc_lo:.4f}')
print(f'  F1-Score           : {f1_m:.4f}  [{f1_lo:.4f},  {f1_hi:.4f}]  w={f1_hi-f1_lo:.4f}')
print(f'  Precision          : {pre_m:.4f}  [{pre_lo:.4f}, {pre_hi:.4f}]  w={pre_hi-pre_lo:.4f}')
print(f'  Recall             : {rec_m:.4f}  [{rec_lo:.4f}, {rec_hi:.4f}]  w={rec_hi-rec_lo:.4f}')
print('═' * 65)
print(f'  n={len(y_true)} | AUC CI width {auc_hi-auc_lo:.4f}')

metrics_df = pd.DataFrame([
    {'metric':'AUC-ROC',   'bootstrap_mean':auc_m, 'bootstrap_lo':auc_lo, 'bootstrap_hi':auc_hi,
     'jackknife_mean':jk_auc_m,'jackknife_lo':jk_auc_lo,'jackknife_hi':jk_auc_hi},
    {'metric':'Accuracy',  'bootstrap_mean':acc_m, 'bootstrap_lo':acc_lo, 'bootstrap_hi':acc_hi,
     'jackknife_mean':None,'jackknife_lo':None,'jackknife_hi':None},
    {'metric':'F1-Score',  'bootstrap_mean':f1_m,  'bootstrap_lo':f1_lo,  'bootstrap_hi':f1_hi,
     'jackknife_mean':None,'jackknife_lo':None,'jackknife_hi':None},
    {'metric':'Precision', 'bootstrap_mean':pre_m, 'bootstrap_lo':pre_lo, 'bootstrap_hi':pre_hi,
     'jackknife_mean':None,'jackknife_lo':None,'jackknife_hi':None},
    {'metric':'Recall',    'bootstrap_mean':rec_m, 'bootstrap_lo':rec_lo, 'bootstrap_hi':rec_hi,
     'jackknife_mean':None,'jackknife_lo':None,'jackknife_hi':None},
])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_metrics_with_ci.csv'), index=False)
print('\n✓ Metrics saved → ensemble_metrics_with_ci.csv')

Computing 95% Bootstrap CIs (n=2000, stratified) + Jackknife CIs...
═════════════════════════════════════════════════════════════════
FINAL ENSEMBLE v3 — CONFIDENCE INTERVALS
═════════════════════════════════════════════════════════════════
  Method: stratified bootstrap (n=2000) + jackknife

  AUC-ROC   bootstrap: 0.8078  [0.7856, 0.8293]  w=0.0437
  AUC-ROC   jackknife: 0.8081  [0.7867, 0.8294]
  Accuracy           : 0.7289  [0.7059, 0.7513]  w=0.0454
  F1-Score           : 0.7334  [0.7104,  0.7553]  w=0.0449
  Precision          : 0.6932  [0.6703, 0.7181]  w=0.0478
  Recall             : 0.7788  [0.7459, 0.8091]  w=0.0632
═════════════════════════════════════════════════════════════════
  n=1520 | AUC CI width 0.0437

✓ Metrics saved → ensemble_metrics_with_ci.csv


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 v3: RESEARCH-GRADE VISUALIZATIONS
# v3 adds: DeLong p-value annotation, jackknife CI, imputation quality plot,
#          method-category legend, AUC gain over best single model
# ═══════════════════════════════════════════════════════════════════════════════

from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(3, 3, figsize=(21, 18))

ALL_COLORS   = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']
model_colors = {col: ALL_COLORS[i % len(ALL_COLORS)] for i, col in enumerate(SCORE_COLS)}
ENS_COLOR    = '#E91E63'
SHORT        = {'P_rPPG':'rPPG','P_efficientnet':'EffNet','P_xception':'Xcep','P_swin':'Swin'}

# Plot 1: ROC curves
ax = axes[0, 0]
for col in SCORE_COLS:
    fpr, tpr, _ = roc_curve(y_true, merged[col].values)
    ax.plot(fpr, tpr, color=model_colors[col], lw=1.5, alpha=0.7,
            label=f'{SHORT.get(col,col)} ({individual_aucs[col]:.3f})')
fpr_e, tpr_e, _ = roc_curve(y_true, best_probs)
ax.plot(fpr_e, tpr_e, color=ENS_COLOR, lw=3, ls='--', label=f'Ensemble ({final_auc:.3f})')
ax.plot([0,1],[0,1],'k:',lw=1,label='Random')
ax.text(0.5, 0.1, f'DeLong p={p_dl:.4f}', ha='center', transform=ax.transAxes,
        fontsize=9, color='darkred' if p_dl < 0.05 else 'gray')
ax.set_xlabel('FPR',fontsize=11); ax.set_ylabel('TPR',fontsize=11)
ax.set_title('ROC Curves + DeLong significance', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)

# Plot 2: PR curves
ax = axes[0, 1]
for col in SCORE_COLS:
    prec_c, rec_c, _ = precision_recall_curve(y_true, merged[col].values)
    ap = average_precision_score(y_true, merged[col].values)
    ax.plot(rec_c, prec_c, color=model_colors[col], lw=1.5, alpha=0.7,
            label=f'{SHORT.get(col,col)} (AP={ap:.3f})')
prec_e, rec_e, _ = precision_recall_curve(y_true, best_probs)
ax.plot(rec_e, prec_e, color=ENS_COLOR, lw=3, ls='--', label=f'Ensemble (AP={final_ap:.3f})')
ax.axhline(y=y_true.mean(), color='gray', ls=':', lw=1, label=f'Baseline ({y_true.mean():.2f})')
ax.set_xlabel('Recall',fontsize=11); ax.set_ylabel('Precision',fontsize=11)
ax.set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Plot 3: Confusion matrix
ax = axes[0, 2]
cm = confusion_matrix(y_true, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], annot_kws={'size':18})
ax.set_xlabel('Predicted',fontsize=11); ax.set_ylabel('Actual',fontsize=11)
ax.set_title(f'Confusion Matrix (t={best_thresh:.3f})', fontsize=12, fontweight='bold')

# Plot 4: Score distribution
ax = axes[1, 0]
ax.hist(best_probs[y_true==0], bins=40, alpha=0.6, color='#4CAF50', label='Real', density=True)
ax.hist(best_probs[y_true==1], bins=40, alpha=0.6, color='#F44336', label='Fake', density=True)
ax.axvline(x=best_thresh, color='black', lw=2, ls='--', label=f'Threshold={best_thresh:.4f}')
ax.set_xlabel('P(Fake)',fontsize=11); ax.set_ylabel('Density',fontsize=11)
ax.set_title('Score Distribution (Best Ensemble)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Plot 5: Calibration curve
ax = axes[1, 1]
for col in SCORE_COLS:
    fp, mp = calibration_curve(y_true, merged[col].values, n_bins=10)
    ax.plot(mp, fp, color=model_colors[col], lw=1.5, alpha=0.6, marker='o', ms=4,
            label=SHORT.get(col,col))
fp_e, mp_e = calibration_curve(y_true, best_probs, n_bins=10)
ax.plot(mp_e, fp_e, color=ENS_COLOR, lw=3, ls='--', marker='s', ms=6, label='Ensemble')
ax.plot([0,1],[0,1],'k:',lw=1,label='Perfect')
ax.set_xlabel('Mean Predicted Prob',fontsize=11); ax.set_ylabel('Fraction Positives',fontsize=11)
ax.set_title('Calibration Curve (Reliability Diagram)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Plot 6: All methods horizontal bar chart
ax = axes[1, 2]
method_names = list(ensemble_results.keys())
method_aucs  = [ensemble_results[m]['auc'] for m in method_names]
_CLOSED_SET  = {'hard_vote','simple_avg','auc_weighted','geometric_mean','logit_avg',
                'harmonic_mean','product_prob','conf_weighted_vote','trimmed_mean',
                'weighted_median','rank_based','borda_count','kemeny_young','power_mean',
                'min_ensemble','max_ensemble'}
bar_c = ['#2196F3' if m in _CLOSED_SET else '#FF5722' for m in method_names]
bar_c = ['#E91E63' if m == best_name else c for m, c in zip(method_names, bar_c)]
ax.barh(range(len(method_names)), method_aucs, color=bar_c, alpha=0.85)
ax.set_yticks(range(len(method_names))); ax.set_yticklabels(method_names, fontsize=7)
ax.axvline(x=max(individual_aucs.values()), color='green', ls='--', lw=1.5,
           label=f'Best single ({max(individual_aucs.values()):.3f})')
ax.set_xlabel('AUC-ROC',fontsize=11)
ax.set_title('All Ensemble Methods v3\n(pink=best, blue=closed, orange=learned)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3)
xl = max(0, min(method_aucs)-0.05)
ax.set_xlim([xl, min(1, max(method_aucs)+0.03)])
for i, auc_v in enumerate(method_aucs):
    ax.text(auc_v+0.001, i, f'{auc_v:.4f}', va='center', fontsize=6)

# Plot 7: Correlation heatmap
ax = axes[2, 0]
corr = merged[SCORE_COLS].corr()
sl = [SHORT.get(c,c) for c in SCORE_COLS]
corr.columns = sl; corr.index = sl
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax,
            square=True, vmin=-1, vmax=1, annot_kws={'size':11})
ax.set_title('Model Correlations (Lower = More Diverse)', fontsize=12, fontweight='bold')

# Plot 8: AUC gain over best individual
ax = axes[2, 1]
names_  = [SHORT.get(c,c) for c in SCORE_COLS] + ['Ensemble']
aucs_   = [individual_aucs[c] for c in SCORE_COLS] + [final_auc]
bars_   = ax.bar(names_, aucs_, color=[model_colors[c] for c in SCORE_COLS]+[ENS_COLOR], alpha=0.85)
best_ind_auc = max(individual_aucs.values())
ax.axhline(y=best_ind_auc, color='gray', ls='--', lw=1, label=f'Best single={best_ind_auc:.4f}')
for bar, auc_v in zip(bars_, aucs_):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{auc_v:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim([max(0,min(aucs_)-0.05), min(1.0,max(aucs_)+0.06)])
ax.set_ylabel('AUC-ROC',fontsize=11)
ax.set_title(f'AUC Gain = +{final_auc - best_ind_auc:.4f} over best single model',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.tick_params(axis='x', rotation=15)

# Plot 9: Bootstrap + jackknife CI comparison
ax = axes[2, 2]
ci_names = ['AUC-ROC','Accuracy','F1-Score','Precision','Recall']
ci_vals  = [auc_m, acc_m, f1_m, pre_m, rec_m]
ci_lo_e  = [auc_m-auc_lo, acc_m-acc_lo, f1_m-f1_lo, pre_m-pre_lo, rec_m-rec_lo]
ci_hi_e  = [auc_hi-auc_m, acc_hi-acc_m, f1_hi-f1_m, pre_hi-pre_m, rec_hi-rec_m]
ax.barh(ci_names, ci_vals, xerr=[ci_lo_e, ci_hi_e],
        color='#2196F3', alpha=0.7, capsize=5, ecolor='#0D47A1', label='Bootstrap')
ax.plot(jk_auc_m, ci_names.index('AUC-ROC'), marker='D', color='#FF5722', ms=8, label='Jackknife AUC', zorder=5)
ax.set_xlabel('Score',fontsize=11)
ax.set_title('Bootstrap 95% CIs + Jackknife', fontsize=12, fontweight='bold')
ax.set_xlim([0.3, 1.05]); ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=9)
for i, (v, lo, hi) in enumerate(zip(ci_vals, ci_lo_e, ci_hi_e)):
    ax.text(v+hi+0.005, i, f'{v:.3f}', va='center', fontsize=9)

active = ' + '.join([SHORT.get(c,c) for c in SCORE_COLS])
plt.suptitle(f'Ensemble Fusion v3 — {active}  |  {final_name}  |  AUC={final_auc:.4f}  |  DeLong p={p_dl:.4f}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, 'ensemble_evaluation_plots_v3.png'),
            dpi=150, bbox_inches='tight', facecolor='white')



print('✓ Plots saved → ensemble_evaluation_plots_v3.png')
plt.close() 

✓ Plots saved → ensemble_evaluation_plots_v3.png


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10 v3: SAVE FINAL PREDICTIONS
# v3 adds: DeLong p-value, jackknife CI, McNemar p-value in output
# ═══════════════════════════════════════════════════════════════════════════════

final_df = merged[['video_id','label'] + SCORE_COLS].copy()

# Imputation flags
if 'columns' in dir(master):
    for col in SCORE_COLS:
        ic = f'{col}_imputed'
        if ic in master.columns:
            final_df[ic] = final_df['video_id'].map(
                dict(zip(master['video_id'], master[ic]))).fillna(False)

for method_name, rd in ensemble_results.items():
    final_df[f'P_{method_name}'] = rd['probs']

final_df['P_final']          = best_probs
final_df['pred_final']       = final_preds
final_df['ensemble_method']  = final_name
final_df['threshold_used']   = best_thresh
final_df['threshold_method'] = "Youden's J"
final_df['temperature_T']    = best_T
final_df['delong_p_value']   = p_dl
final_df['mcnemar_p_value']  = mc_p

out_path = os.path.join(OUTPUT_DIR, 'ensemble_final_predictions_v3.csv')
final_df.to_csv(out_path, index=False)

print('═' * 70)
print('FINAL OUTPUT v3 SAVED')
print('═' * 70)
print(f'  File    : {out_path}')
print(f'  Rows    : {len(final_df)}')
print(f'  Columns : {len(final_df.columns)}')
print()
print('  Key columns:')
print('    P_final          → Best ensemble probability')
print('    pred_final       → Binary prediction (0=Real, 1=Fake)')
print(f'    threshold_used   → {best_thresh:.4f}')
print(f'    temperature_T    → {best_T:.3f}')
print(f'    delong_p_value   → {p_dl:.4f}  (ensemble vs best single model)')
print(f'    mcnemar_p_value  → {mc_p:.4f}')
print('=' * 70)
print()
print('FINAL SUMMARY TABLE (sorted by AUC)')
print('─' * 72)
print(f"{'Model/Method':38s} {'AUC':>8}  Type")
print('─' * 72)
for col in SCORE_COLS:
    print(f"  Individual: {MODEL_LABELS[col]:26s} {individual_aucs[col]:.4f}  base model")
print('─' * 50)
_OOF_K = {'meta_lr','meta_mlp','meta_svc','meta_gbm','meta_rf','meta_et','meta_lgbm',
           'meta_catboost','meta_xgb','optuna_multi_obj','frank_wolfe_weights',
           'swa_weights','mixture_of_experts','des_knn','super_ensemble_L2','venn_abers_final'}
for method, res in sorted(ensemble_results.items(), key=lambda x: -x[1]['auc']):
    mark = ' ★ BEST' if method == best_name else ''
    typ  = 'OOF learned' if method in _OOF_K else 'closed-form'
    print(f"  Ensemble:   {method:28s} {res['auc']:.4f}  {typ}{mark}")
print('─' * 72)
print(f'\n  AUC gain over best single model: +{final_auc - max(individual_aucs.values()):.4f}')
print(f'  DeLong significance: p={p_dl:.4f}  McNemar: p={mc_p:.4f}')




══════════════════════════════════════════════════════════════════════
FINAL OUTPUT v3 SAVED
══════════════════════════════════════════════════════════════════════
  File    : /kaggle/working/ensemble_final_predictions_v3.csv
  Rows    : 1520
  Columns : 49

  Key columns:
    P_final          → Best ensemble probability
    pred_final       → Binary prediction (0=Real, 1=Fake)
    threshold_used   → 0.4377
    temperature_T    → 0.711
    delong_p_value   → 0.2168  (ensemble vs best single model)
    mcnemar_p_value  → 0.4504

FINAL SUMMARY TABLE (sorted by AUC)
────────────────────────────────────────────────────────────────────────
Model/Method                                AUC  Type
────────────────────────────────────────────────────────────────────────
  Individual: rPPG (ML Stacking)         0.6283  base model
  Individual: EfficientNet-B4 (BiLSTM+Attn) 0.7784  base model
  Individual: Swin-Tiny (BiLSTM+DCT)     0.7884  base model
───────────────────────────────────────────────

In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11 — SAVE ENSEMBLE MODEL FOR BACKEND INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
#
# Saves a self-contained model artifact: ensemble_model.pkl + ensemble_model.json
# Your backend loads ONE file and calls predict(p_rppg, p_eff, p_swin) → label+confidence
#
# Architecture of the saved model:
#   Input : 3 raw probabilities from base models (floats in [0,1])
#   Step 1: Weighted sum  (Optuna weights, AUC=0.8080)
#   Step 2: Temperature scaling  (T learned OOF, monotonic → AUC unchanged)
#   Step 3: Threshold  (Youden's J = maximises TPR-FPR)
#   Output: "FAKE" / "REAL" + confidence %
# ═══════════════════════════════════════════════════════════════════════════════

import joblib, json as _json
from scipy.special import expit, logit

# ─── 1. Collect all state needed for inference ───────────────────────────────
# Map SCORE_COLS → base model names for the backend to know the input order
_COL_TO_MODEL = {
    'P_rPPG'        : 'rppg',
    'P_efficientnet': 'efficientnet',
    'P_xception'    : 'xception',
    'P_swin'        : 'swin',
}
input_model_names = [_COL_TO_MODEL[c] for c in SCORE_COLS]


model_artifact = {
    # ── Identity ──────────────────────────────────────────────────────────────
    'version'          : 'v5',                   # ★ FIX: was 'v4'
    'ensemble_method'  : str(final_name),
    'best_strategy'    : str(best_strategy),

    # ── Input spec ────────────────────────────────────────────────────────────
    'input_model_names': input_model_names,
    'score_cols'       : list(SCORE_COLS),
    'n_models'         : len(SCORE_COLS),

    # ── Ensemble weights ──────────────────────────────────────────────────────
    'optuna_weights'      : best_w_opt.tolist(),
    'bma_weights'         : bma_weights.tolist(),
    'slsqp_weights'       : fw_w.tolist(),        # ★ FIX: renamed from 'frank_wolfe_weights'

    # ── Calibration ───────────────────────────────────────────────────────────
    'temperature_T'    : float(best_T),

    # ── Decision thresholds ───────────────────────────────────────────────────
    'threshold_youden' : float(best_thresh),
    'threshold_f2'     : float(best_f2_thresh),
    'threshold_cv_f1'  : float(cv_thresh),

    # ── Performance ───────────────────────────────────────────────────────────
    'auc_roc'          : float(final_auc),
    'individual_aucs'  : {c: float(v) for c,v in individual_aucs.items()},
    'model_labels'     : dict(MODEL_LABELS),
    'n_train_samples'  : int(N),
    'delong_p_value'   : float(p_dl),
}

# ─── 2. Save as .pkl (joblib) — primary format for Python backends ────────────
pkl_path = os.path.join(OUTPUT_DIR, 'ensemble_model.pkl')
joblib.dump(model_artifact, pkl_path, compress=3)
print(f'✓ Saved ensemble_model.pkl  ({os.path.getsize(pkl_path)/1024:.1f} KB)')

# ─── 3. Save as .json — for inspection and non-Python backends ───────────────
json_path = os.path.join(OUTPUT_DIR, 'ensemble_model.json')
with open(json_path, 'w') as f:
    _json.dump(model_artifact, f, indent=2)
print(f'✓ Saved ensemble_model.json  ({os.path.getsize(json_path)/1024:.1f} KB)')

# ─── 4. Verification: round-trip reload and spot-check ───────────────────────
loaded = joblib.load(pkl_path)
weights_ok   = abs(sum(loaded['optuna_weights']) - 1.0) < 1e-6
threshold_ok = 0 < loaded['threshold_youden'] < 1
input_ok     = loaded['input_model_names'] == input_model_names
print(f'\n  Verification: weights_sum={sum(loaded["optuna_weights"]):.6f}  threshold={loaded["threshold_youden"]:.4f}  inputs={loaded["input_model_names"]}')
assert weights_ok and threshold_ok and input_ok, "Model artifact verification failed!"
print('  ✓ Artifact verified — round-trip load successful')

# ─── 5. Print backend usage instructions ─────────────────────────────────────
print()
print('═' * 70)
print('BACKEND INFERENCE — COPY THIS INTO YOUR WEBSITE BACKEND')
print('═' * 70)
print(f"""
# backend_predict.py
import joblib, numpy as np
from scipy.special import expit, logit

# Load ONCE at server startup
MODEL = joblib.load('ensemble_model.pkl')
W     = np.array(MODEL['optuna_weights'])     # {best_w_opt.tolist()}
T     = MODEL['temperature_T']                 # {float(best_T):.4f}
THR   = MODEL['threshold_youden']              # {float(best_thresh):.4f}  ← use this
THR_F2= MODEL['threshold_f2']                 # {float(best_f2_thresh):.4f}  ← if recall matters more
EPS   = 1e-7

# Input ORDER must match: {input_model_names}
def predict(p_{'_'.join(input_model_names)}_list):
    probs   = np.array(p_{'_'.join(input_model_names)}_list, dtype=float).clip(EPS, 1-EPS)
    p_final = float(W @ probs)                 # weighted ensemble
    if T != 1.0:                               # apply temperature calibration
        p_final = float(expit(logit(np.clip(p_final, EPS, 1-EPS)) / T))
    is_fake    = p_final >= THR
    confidence = p_final if is_fake else (1.0 - p_final)
    return {{
        "label"     : "FAKE" if is_fake else "REAL",
        "confidence": round(confidence * 100, 1),  # e.g. 78.3
        "p_fake"    : round(p_final, 4),
        "threshold" : THR,
    }}

# Example call:
# result = predict([p_rppg, p_efficientnet, p_swin])
# → {{"label": "FAKE", "confidence": 78.3, "p_fake": 0.8312, "threshold": {float(best_thresh):.4f}}}
""")
print('═' * 70)
print(f'  Model AUC-ROC : {final_auc:.4f}')
print(f'  Input order   : {input_model_names}')
print(f'  Weights       : {[round(w,4) for w in best_w_opt.tolist()]}')
print(f'  Threshold     : {float(best_thresh):.4f} (Youden) | {float(best_f2_thresh):.4f} (F2)')
print(f'  Temperature T : {float(best_T):.4f}')
print('═' * 70)


✓ Saved ensemble_model.pkl  (0.6 KB)
✓ Saved ensemble_model.json  (1.1 KB)

  Verification: weights_sum=1.000000  threshold=0.4377  inputs=['rppg', 'efficientnet', 'swin']
  ✓ Artifact verified — round-trip load successful

══════════════════════════════════════════════════════════════════════
BACKEND INFERENCE — COPY THIS INTO YOUR WEBSITE BACKEND
══════════════════════════════════════════════════════════════════════

# backend_predict.py
import joblib, numpy as np
from scipy.special import expit, logit

# Load ONCE at server startup
MODEL = joblib.load('ensemble_model.pkl')
W     = np.array(MODEL['optuna_weights'])     # [0.10633995853969735, 0.34351599209359385, 0.5501440493667088]
T     = MODEL['temperature_T']                 # 0.7109
THR   = MODEL['threshold_youden']              # 0.4377  ← use this
THR_F2= MODEL['threshold_f2']                 # 0.1600  ← if recall matters more
EPS   = 1e-7

# Input ORDER must match: ['rppg', 'efficientnet', 'swin']
def predict(p_rppg_efficient

In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12 v4: ARCHIVE OUTPUTS
# ★ FIX: Archive only specific output files, NOT all of /kaggle/working
#   Old code: shutil.make_archive('...', 'zip', OUTPUT_DIR)
#   Problem:  Zips the entire working dir including the notebook itself
#             (with embedded plot outputs), prior run artifacts, and on the
#             second Kaggle run it zips the zip into itself — causing disk-full
# ═══════════════════════════════════════════════════════════════════════════════

import shutil, zipfile

# Define exactly which files to archive
OUTPUT_FILES = [
    'ensemble_model.pkl',
    'ensemble_model.json',
    'ensemble_final_predictions_v3.csv',
    'ensemble_metrics_with_ci.csv',
    'ensemble_evaluation_plots_v3.png',
    'rppg_predictions_fixed.csv',
]

zip_path = os.path.join(OUTPUT_DIR, 'ensemble_outputs_v3.zip')

# Remove stale zip from prior run to avoid self-inclusion
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in OUTPUT_FILES:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=fname)
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  ✓ Added {fname:50s}  {size_kb:8.1f} KB')
        else:
            print(f'  ⚠ Skipped (not found): {fname}')

zip_size_kb = os.path.getsize(zip_path) / 1024
print(f'\n✓ Archive saved → ensemble_outputs_v3.zip  ({zip_size_kb:.1f} KB)')
print()
print('All files in /kaggle/working/:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:55s}  {size_kb:8.1f} KB')

  ✓ Added ensemble_model.pkl                                       0.6 KB
  ✓ Added ensemble_model.json                                      1.1 KB
  ✓ Added ensemble_final_predictions_v3.csv                     1275.8 KB
  ✓ Added ensemble_metrics_with_ci.csv                             0.5 KB
  ✓ Added ensemble_evaluation_plots_v3.png                       650.4 KB
  ✓ Added rppg_predictions_fixed.csv                              64.0 KB

✓ Archive saved → ensemble_outputs_v3.zip  (1021.4 KB)

All files in /kaggle/working/:
  __notebook__.ipynb                                          145.6 KB
  catboost_info                                                 4.0 KB
  ensemble_evaluation_plots_v3.png                            650.4 KB
  ensemble_final_predictions_v3.csv                          1275.8 KB
  ensemble_metrics_with_ci.csv                                  0.5 KB
  ensemble_model.json                                           1.1 KB
  ensemble_model.pkl                      